[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus/05_indefinite_and_definite_integrals/exercises.ipynb)

# Module 05 — Indefinite and Definite Integrals: Exercises

Forty fully solved problems across four tiers. Every numeric or algorithmic answer is
recomputed by the code cell that follows its solution, so no boxed value here rests on memory.

Run this preamble first; every later code cell depends on it.

In [1]:
import numpy as np
import sympy as sp
from scipy import integrate

rng = np.random.default_rng(0)
np.set_printoptions(precision=6, suppress=True)

x, t, u, y = sp.symbols("x t u y", real=True)


def close(a, b, tol=1e-9):
    """Report agreement between a computed value and a claimed boxed value."""
    a, b = float(a), float(b)
    ok = abs(a - b) <= tol * max(1.0, abs(b))
    print(f"  computed {a:.12f}   boxed {b:.12f}   |diff| = {abs(a - b):.2e}   {'OK' if ok else 'MISMATCH'}")
    assert ok, (a, b)
    return ok

The preamble imports NumPy, SymPy and SciPy quadrature, and defines `close`, which prints a
computed value next to the boxed one and asserts they agree.

## L0 — Concept Checks

### Problem L0.1 — Why a Positive Integrand Cannot Give $-2$
**Source Attribution:** Spivak, *Calculus* (4th Ed.), Chapter 14.

**Statement.**

Identify the mathematical fallacy in the following evaluation:

$$
\int_{-1}^1 \frac{1}{x^2} \, dx = \left[ -\frac{1}{x} \right]_{-1}^1 = \left(-\frac{1}{1}\right) - \left(-\frac{1}{-1}\right) = -1 - 1 = -2
$$

Why is this result absurd?

**Intuition.**
The integrand $f(x) = \frac{1}{x^2}$ is strictly positive ($f(x) \gt 0$) for all $x \neq 0$. The definite integral of a strictly positive function must be positive. A negative result ($-2$) indicates a violation of hypotheses.

**Solution.**
1. **Hypothesis Verification of FTC II**: The Fundamental Theorem of Calculus Part II states that if $g'(x) = f(x)$ on $[a, b]$, then $\int_a^b f(x) dx = g(b) - g(a)$. However, this requires $f(x)$ to be bounded and integrable on $[a, b]$, and $g(x)$ to be continuous on $[a, b]$.
2. **Failure of Hypotheses**: $f(x) = x^{-2}$ has an essential infinite discontinuity at $x = 0 \in [-1, 1]$, making $f$ unbounded on $[-1, 1]$. Furthermore, $g(x) = -\frac{1}{x}$ is discontinuous at $x = 0$.
3. **Improper Integral Treatment**: The integral must be split at $x = 0$ as an improper integral:

$$
\int_{-1}^1 \frac{1}{x^2} \, dx = \lim_{\varepsilon_1 \to 0^+} \int_{-1}^{-\varepsilon_1} \frac{1}{x^2} \, dx + \lim_{\varepsilon_2 \to 0^+} \int_{\varepsilon_2}^1 \frac{1}{x^2} \, dx
$$

Evaluating the first limit:

$$
\lim_{\varepsilon_1 \to 0^+} \left[ -\frac{1}{x} \right]_{-1}^{-\varepsilon_1} = \lim_{\varepsilon_1 \to 0^+} \left( \frac{1}{\varepsilon_1} - 1 \right) = +\infty
$$

Thus, the integral diverges to $+\infty$.

$$
\boxed{\text{Fallacy: FTC II requires continuity on } [-1, 1]. \text{ The integral diverges to } +\infty.}
$$

**Key takeaway.**
Blindly plugging limits into antiderivative formulas without verifying domain continuity causes severe sign errors and invalid infinite results.

In [2]:
print("Problem L0.1 - the fallacious value versus the honest truncation")
g_bad = lambda z: -1.0 / z
blind = g_bad(1.0) - g_bad(-1.0)
print(f"  blind FTC II gives {blind:+.4f}, yet 1/x^2 > 0 for every x != 0 in [-1, 1]")
for eps in (1e-2, 1e-4, 1e-6):
    print(f"  int over |x| in [{eps:g}, 1] of x^-2 dx = {2 * (1 / eps - 1):16.2f}")
print("  the truncations grow without bound, so the integral diverges to +infinity")
assert blind < 0

Problem L0.1 - the fallacious value versus the honest truncation
  blind FTC II gives -2.0000, yet 1/x^2 > 0 for every x != 0 in [-1, 1]
  int over |x| in [0.01, 1] of x^-2 dx =           198.00
  int over |x| in [0.0001, 1] of x^-2 dx =         19998.00
  int over |x| in [1e-06, 1] of x^-2 dx =       1999998.00
  the truncations grow without bound, so the integral diverges to +infinity


### Problem L0.2 — Net Signed Area of a Straight Line
**Source Attribution:** Stewart, *Calculus: Early Transcendentals* (8th Ed.), Chapter 5.

**Statement.**

Calculate $\int_0^4 (2x - 4) \, dx$ using geometric area formulas, and verify with FTC II.

**Intuition.**
The definite integral computes net signed area: region above the $x$-axis contributes positive area, while region below the $x$-axis contributes negative area. $2x - 4$ crosses the $x$-axis at $x = 2$.

**Solution.**
1. **Geometric Solution**:
   - For $x \in [0, 2]$, $2x - 4 \le 0$. Triangle 1 has vertices $(0,0), (2,0), (0,-4)$.  
     $\text{Area}_1 = \frac{1}{2} \times \text{base} \times \text{height} = \frac{1}{2} \times 2 \times 4 = 4$.  
     Since it lies below the axis, $\text{Signed Area}_1 = -4$.
   - For $x \in [2, 4]$, $2x - 4 \ge 0$. Triangle 2 has vertices $(2,0), (4,0), (4,4)$.  
     $\text{Area}_2 = \frac{1}{2} \times 2 \times 4 = 4$.  
     Since it lies above the axis, $\text{Signed Area}_2 = +4$.
   - Net Signed Area $= -4 + 4 = 0$.

2. **FTC II Solution**:

$$
\int_0^4 (2x - 4) \, dx = \left[ x^2 - 4x \right]_0^4 = (4^2 - 4(4)) - (0^2 - 4(0)) = (16 - 16) - 0 = 0
$$

$$
\boxed{0}
$$

**Key takeaway.**
The definite integral measures *net signed area*, allowing continuous balance between positive and negative region contributions.

In [3]:
print("Problem L0.2 - net signed area of 2x - 4 on [0, 4]")
val, _ = integrate.quad(lambda z: 2 * z - 4, 0.0, 4.0)
close(val, 0.0, tol=1e-12)
below, _ = integrate.quad(lambda z: 2 * z - 4, 0.0, 2.0)
above, _ = integrate.quad(lambda z: 2 * z - 4, 2.0, 4.0)
print(f"  signed pieces: {below:+.4f} on [0,2] and {above:+.4f} on [2,4]")

Problem L0.2 - net signed area of 2x - 4 on [0, 4]
  computed 0.000000000000   boxed 0.000000000000   |diff| = 5.63e-17   OK
  signed pieces: -4.0000 on [0,2] and +4.0000 on [2,4]


### Problem L0.3 — Accumulation with a Composite Upper Limit
**Source Attribution:** Spivak, *Calculus* (4th Ed.), Chapter 14.

**Statement.**

Compute $G'(x)$ where $G(x) = \int_0^{x^2} \sin(t^2) \, dt$.

**Intuition.**
By FTC Part I, the derivative of $\int_0^u \sin(t^2) dt$ with respect to $u$ is $\sin(u^2)$. Since the upper limit is $u(x) = x^2$, we must apply the chain rule.

**Solution.**
Define $F(u) = \int_0^u \sin(t^2) \, dt$. Then $G(x) = F(x^2)$.  
By FTC Part I:

$$
F'(u) = \frac{d}{du} \int_0^u \sin(t^2) \, dt = \sin(u^2)
$$

By the Chain Rule:

$$
G'(x) = \frac{d}{dx} F(x^2) = F'(x^2) \cdot \frac{d}{dx}(x^2) = \sin\left((x^2)^2\right) \cdot 2x = 2x \sin(x^4)
$$

$$
\boxed{2x \sin(x^4)}
$$

**Key takeaway.**
Combining FTC I with the Chain Rule allows immediate differentiation of accumulation functions with composite limit functions.

In [4]:
print("Problem L0.3 - d/dx int_0^{x^2} sin(t^2) dt at x = 1.3")
G = lambda z: integrate.quad(lambda s: np.sin(s**2), 0.0, z**2)[0]
x0, h = 1.3, 1e-5
close((G(x0 + h) - G(x0 - h)) / (2 * h), 2 * x0 * np.sin(x0**4), tol=1e-6)

Problem L0.3 - d/dx int_0^{x^2} sin(t^2) dt at x = 1.3
  computed 0.732238512235   boxed 0.732238514579   |diff| = 2.34e-09   OK


True

### Problem L0.4 — The Constant Is Per Connected Component
**Source Attribution:** Apostol, *Calculus, Volume I* (2nd Ed.), Chapter 5.

**Statement.**

Find the most general antiderivative of $f(x) = \frac{1}{x^2}$ on its full domain $\mathbb{R} \setminus \{0\}$.

**Intuition.**
The domain of $f(x)$ consists of two disconnected open intervals: $(-\infty, 0)$ and $(0, \infty)$. The constant of integration in an antiderivative is constant on connected components, but can take *different* values on disconnected components.

**Solution.**
On $(0, \infty)$, $\frac{d}{dx}\left(-\frac{1}{x} + C_1\right) = \frac{1}{x^2}$.  
On $(-\infty, 0)$, $\frac{d}{dx}\left(-\frac{1}{x} + C_2\right) = \frac{1}{x^2}$.  
Thus, the most general antiderivative $F(x)$ is:

$$
F(x) = \begin{cases} -\frac{1}{x} + C_1 & \text{if } x \gt 0 \\ -\frac{1}{x} + C_2 & \text{if } x \lt 0 \end{cases}
$$

where $C_1, C_2 \in \mathbb{R}$ are independent constants.

$$
\boxed{F(x) = -\frac{1}{x} + C_1 \text{ for } x \gt 0; \quad -\frac{1}{x} + C_2 \text{ for } x \lt 0}
$$

**Key takeaway.**
The "plus $C$" constant rule applies strictly to connected domain intervals. Disconnected domain components admit independent arbitrary constants.

In [5]:
print("Problem L0.4 - -1/x is an antiderivative of 1/x^2 on each component")
print("  d/dx(-1/x) =", sp.simplify(sp.diff(-1 / x, x)))
assert sp.simplify(sp.diff(-1 / x, x) - 1 / x**2) == 0
step = sp.Piecewise((0, x < 0), (1, True))
print("  a locally constant, globally non-constant function on R \\ {0}: ", step)
print("  its derivative vanishes on both components, so C1 and C2 are independent")

Problem L0.4 - -1/x is an antiderivative of 1/x^2 on each component


  d/dx(-1/x) = x**(-2)
  a locally constant, globally non-constant function on R \ {0}:  Piecewise((0, x < 0), (1, True))
  its derivative vanishes on both components, so C1 and C2 are independent


### Problem L0.5 — Odd Symmetry Kills a Definite Integral
**Source Attribution:** MIT OpenCourseWare 18.01.

**Statement.**

Prove that if $f(x)$ is an odd function ($f(-x) = -f(x)$) continuous on $[-a, a]$, then $\int_{-a}^a f(x) \, dx = 0$.

**Intuition.**
An odd function has rotational symmetry about the origin. The area accumulated on $[-a, 0]$ has identical magnitude but opposite sign to the area accumulated on $[0, a]$.

**Solution.**
Split the integral into two subintervals:

$$
\int_{-a}^a f(x) \, dx = \int_{-a}^0 f(x) \, dx + \int_0^a f(x) \, dx
$$

In the first integral $\int_{-a}^0 f(x) \, dx$, substitute $u = -x \implies dx = -du$.  
When $x = -a$, $u = a$; when $x = 0$, $u = 0$:

$$
\int_{-a}^0 f(x) \, dx = \int_a^0 f(-u) (-du) = \int_0^a f(-u) \, du
$$

Since $f$ is odd, $f(-u) = -f(u)$:

$$
\int_0^a f(-u) \, du = \int_0^a (-f(u)) \, du = -\int_0^a f(u) \, du
$$

Substituting this back into the total sum:

$$
\int_{-a}^a f(x) \, dx = -\int_0^a f(u) \, du + \int_0^a f(x) \, dx = 0
$$

$$
\boxed{0}
$$

**Key takeaway.**
Symmetry transformations dramatically simplify definite integrals by cancelling equal-and-opposite signed areas without requiring antiderivatives.

In [6]:
print("Problem L0.5 - odd integrands over a symmetric interval")
for f_odd, name in ((lambda z: z**3 - 2 * z, "x^3 - 2x"),
                    (lambda z: z**2 * np.sin(z), "x^2 sin x")):
    val, _ = integrate.quad(f_odd, -1.7, 1.7)
    print(f"  int_-1.7^1.7 ({name}) dx:")
    close(val, 0.0, tol=1e-10)

Problem L0.5 - odd integrands over a symmetric interval
  int_-1.7^1.7 (x^3 - 2x) dx:
  computed 0.000000000000   boxed 0.000000000000   |diff| = 0.00e+00   OK
  int_-1.7^1.7 (x^2 sin x) dx:
  computed 0.000000000000   boxed 0.000000000000   |diff| = 0.00e+00   OK


## L1 — Foundations

### Problem L1.1 — A Riemann Sum Limit That Is an Arctangent
**Source Attribution:** Spivak, *Calculus* (4th Ed.), Chapter 13 / Stewart Ch. 5.

**Statement.**

Evaluate the limit of the sum:

$$
L = \lim_{n \to \infty} \sum_{i=1}^n \frac{n}{n^2 + i^2}
$$

**Intuition.**
Algebraically factor $n^2$ from the denominator to reveal the standard right-endpoint Riemann sum form $\frac{1}{n} \sum_{i=1}^n f\left(\frac{i}{n}\right)$.

**Solution.**
Rewrite the general term of the sum:

$$
\frac{n}{n^2 + i^2} = \frac{n}{n^2 \left(1 + \left(\frac{i}{n}\right)^2\right)} = \frac{1}{n} \cdot \frac{1}{1 + \left(\frac{i}{n}\right)^2}
$$

Thus:

$$
L = \lim_{n \to \infty} \frac{1}{n} \sum_{i=1}^n \frac{1}{1 + \left(\frac{i}{n}\right)^2}
$$

This matches the tagged Riemann sum for $f(x) = \frac{1}{1 + x^2}$ on $[0, 1]$ with subinterval width $\Delta x = \frac{1}{n}$ and right endpoints $x_i = \frac{i}{n}$:

$$
L = \int_0^1 \frac{1}{1 + x^2} \, dx = \Big[ \arctan x \Big]_0^1 = \arctan(1) - \arctan(0) = \frac{\pi}{4}
$$

$$
\boxed{\frac{\pi}{4}}
$$

**Key takeaway.**
Infinite series involving structural ratios $\frac{i}{n}$ and $\frac{1}{n}$ can be evaluated exactly by translating them into definite Riemann integrals.

In [7]:
print("Problem L1.1 - lim_n sum_{i=1}^n n/(n^2 + i^2)")
partial = None
for n in (10**3, 10**5, 10**6):
    i = np.arange(1, n + 1, dtype=float)
    partial = float((n / (n**2 + i**2)).sum())
    print(f"  n = {n:>8}: partial sum = {partial:.10f}")
close(partial, np.pi / 4, tol=1e-6)

Problem L1.1 - lim_n sum_{i=1}^n n/(n^2 + i^2)
  n =     1000: partial sum = 0.7851481217
  n =   100000: partial sum = 0.7853956634
  n =  1000000: partial sum = 0.7853979134
  computed 0.785397913397   boxed 0.785398163397   |diff| = 2.50e-07   OK


True

### Problem L1.2 — Substitution Under a Square Root
**Source Attribution:** Demidovich, *Problems in Mathematical Analysis*, No. 1152.

**Statement.**

Compute the indefinite integral:

$$
I = \int \frac{x}{\sqrt{1 + x^2}} \, dx
$$

**Intuition.**
Notice that the numerator $x \, dx$ is proportional to the differential of the inner expression $1 + x^2$. A single $u$-substitution handles the square root cleanly.

**Solution.**
Let $u = 1 + x^2 \implies du = 2x \, dx \implies x \, dx = \frac{1}{2} du$.  
Substitute into the integral:

$$
I = \int \frac{\frac{1}{2} du}{\sqrt{u}} = \frac{1}{2} \int u^{-\frac{1}{2}} \, du = \frac{1}{2} \cdot \frac{u^{\frac{1}{2}}}{\frac{1}{2}} + C = \sqrt{u} + C
$$

Substitute back $u = 1 + x^2$:

$$
I = \sqrt{1 + x^2} + C
$$

$$
\boxed{\sqrt{1 + x^2} + C}
$$

**Key takeaway.**
Recognizing derivative-pair relationships $f(g(x))g'(x)$ simplifies algebraic integrand powers into standard polynomial integrals.

In [8]:
print("Problem L1.2 - differentiate the claimed antiderivative")
F = sp.sqrt(1 + x**2)
print("  d/dx sqrt(1+x^2) =", sp.simplify(sp.diff(F, x)))
assert sp.simplify(sp.diff(F, x) - x / sp.sqrt(1 + x**2)) == 0
print("  matches the integrand, so the boxed answer is correct")

Problem L1.2 - differentiate the claimed antiderivative
  d/dx sqrt(1+x^2) = x/sqrt(x**2 + 1)
  matches the integrand, so the boxed answer is correct


### Problem L1.3 — Repeated Parts on a Polynomial-Exponential
**Source Attribution:** Demidovich, *Problems in Mathematical Analysis*, No. 1205.

**Statement.**

Evaluate the indefinite integral:

$$
I = \int x^2 e^{3x} \, dx
$$

**Intuition.**
The integrand is a product of a algebraic polynomial $x^2$ and an exponential function $e^{3x}$. Repeated integration by parts reduces the degree of $x^2$ step-by-step to zero.

**Solution.**
Use the tabular / repeated IBP technique ($\int u \, dv = uv - \int v \, du$).

**First IBP:**
Let $u_1 = x^2 \implies du_1 = 2x \, dx$.  
Let $dv_1 = e^{3x} dx \implies v_1 = \frac{1}{3} e^{3x}$.

$$
I = \frac{1}{3} x^2 e^{3x} - \int \frac{1}{3} e^{3x} (2x) \, dx = \frac{1}{3} x^2 e^{3x} - \frac{2}{3} \int x e^{3x} \, dx
$$

**Second IBP:**
For $\int x e^{3x} dx$, let $u_2 = x \implies du_2 = dx$, and $dv_2 = e^{3x} dx \implies v_2 = \frac{1}{3} e^{3x}$.

$$
\int x e^{3x} \, dx = \frac{1}{3} x e^{3x} - \int \frac{1}{3} e^{3x} \, dx = \frac{1}{3} x e^{3x} - \frac{1}{9} e^{3x}
$$

**Combine results:**

$$
\begin{aligned}
I &= \frac{1}{3} x^2 e^{3x} - \frac{2}{3} \left( \frac{1}{3} x e^{3x} - \frac{1}{9} e^{3x} \right) + C \\
&= \frac{1}{3} x^2 e^{3x} - \frac{2}{9} x e^{3x} + \frac{2}{27} e^{3x} + C = e^{3x} \left( \frac{x^2}{3} - \frac{2x}{9} + \frac{2}{27} \right) + C
\end{aligned}
$$

$$
\boxed{e^{3x} \left( \frac{x^2}{3} - \frac{2x}{9} + \frac{2}{27} \right) + C}
$$

**Key takeaway.**
Integrating polynomial-exponential products repeatedly reduces the algebraic power using the LIATE/IBP rule until a basic exponential integral remains.

In [9]:
print("Problem L1.3 - differentiate the claimed antiderivative")
F = sp.exp(3 * x) * (x**2 / 3 - 2 * x / 9 + sp.Rational(2, 27))
print("  d/dx F =", sp.simplify(sp.diff(F, x)))
assert sp.simplify(sp.diff(F, x) - x**2 * sp.exp(3 * x)) == 0

Problem L1.3 - differentiate the claimed antiderivative
  d/dx F = x**2*exp(3*x)


### Problem L1.4 — Partial Fractions with Distinct Linear Factors
**Source Attribution:** Stewart, *Calculus: Early Transcendentals* (8th Ed.), Chapter 7.

**Statement.**

Compute:

$$
I = \int \frac{5x - 3}{x^2 - 3x + 2} \, dx
$$

**Intuition.**
The denominator factorizes into distinct linear factors $(x - 1)(x - 2)$. Partial fraction decomposition splits the rational function into simple logarithms.

**Solution.**
Factor the denominator: $x^2 - 3x + 2 = (x - 1)(x - 2)$.  
Set up the PFD:

$$
\frac{5x - 3}{(x - 1)(x - 2)} = \frac{A}{x - 1} + \frac{B}{x - 2}
$$

Multiply through by $(x - 1)(x - 2)$:

$$
5x - 3 = A(x - 2) + B(x - 1)
$$

Solve for coefficients:
- Set $x = 1$: $5(1) - 3 = A(1 - 2) + B(0) \implies 2 = -A \implies A = -2$.
- Set $x = 2$: $5(2) - 3 = A(0) + B(2 - 1) \implies 7 = B \implies B = 7$.

Integrate the terms:

$$
I = \int \left( \frac{-2}{x - 1} + \frac{7}{x - 2} \right) \, dx = -2 \ln \lvert x - 1 \rvert + 7 \ln \lvert x - 2 \rvert + C
$$

$$
\boxed{7 \ln \lvert x - 2 \rvert - 2 \ln \lvert x - 1 \rvert + C}
$$

**Key takeaway.**
Partial fraction decomposition converts complex rational integrals into elementary linear logarithmic components.

In [10]:
print("Problem L1.4 - check the decomposition and then the antiderivative")
lhs = (5 * x - 3) / (x**2 - 3 * x + 2)
rhs = -2 / (x - 1) + 7 / (x - 2)
assert sp.simplify(lhs - rhs) == 0
print("  partial fractions identity verified:  A = -2,  B = 7")
F = 7 * sp.log(x - 2) - 2 * sp.log(x - 1)      # valid branch: x > 2
assert sp.simplify(sp.diff(F, x) - lhs) == 0
print("  d/dx F reproduces the integrand on x > 2")

Problem L1.4 - check the decomposition and then the antiderivative


  partial fractions identity verified:  A = -2,  B = 7


  d/dx F reproduces the integrand on x > 2


### Problem L1.5 — A Quarter Circle by Trigonometric Substitution
**Source Attribution:** MIT Integration Bee 2019 / Stewart Ch. 7.

**Statement.**

Evaluate:

$$
I = \int_0^1 \sqrt{1 - x^2} \, dx
$$

**Intuition.**
$y = \sqrt{1 - x^2}$ represents the upper half of the unit circle $x^2 + y^2 = 1$. The integral from $0$ to $1$ computes the area of a quarter circle of radius 1. Alternatively, trigonometric substitution $x = \sin \theta$ eliminates the square root.

**Solution.**
**Trigonometric Substitution Method**:  
Let $x = \sin \theta \implies dx = \cos \theta \, d\theta$.  
Limits: when $x = 0$, $\theta = 0$; when $x = 1$, $\theta = \frac{\pi}{2}$.  
Since $\sqrt{1 - \sin^2 \theta} = \cos \theta$ for $\theta \in [0, \frac{\pi}{2}]$:

$$
I = \int_0^{\frac{\pi}{2}} \cos \theta \cdot (\cos \theta \, d\theta) = \int_0^{\frac{\pi}{2}} \cos^2 \theta \, d\theta
$$

Use the half-angle identity $\cos^2 \theta = \frac{1 + \cos(2\theta)}{2}$:

$$
I = \int_0^{\frac{\pi}{2}} \frac{1 + \cos(2\theta)}{2} \, d\theta = \left[ \frac{\theta}{2} + \frac{\sin(2\theta)}{4} \right]_0^{\frac{\pi}{2}} = \left( \frac{\pi}{4} + 0 \right) - (0 + 0) = \frac{\pi}{4}
$$

$$
\boxed{\frac{\pi}{4}}
$$

**Key takeaway.**
Trigonometric substitutions exploit Pythagorean identities ($\sin^2 \theta + \cos^2 \theta = 1$) to convert radical expressions $\sqrt{a^2 - x^2}$ into standard trigonometric integrals.

In [11]:
print("Problem L1.5 - quarter of the unit disc")
val, _ = integrate.quad(lambda z: np.sqrt(1 - z**2), 0.0, 1.0)
close(val, np.pi / 4)

Problem L1.5 - quarter of the unit disc
  computed 0.785398163397   boxed 0.785398163397   |diff| = 2.22e-16   OK


True

### Problem L1.6 — Weierstrass Substitution on a Trigonometric Fraction
**Source Attribution:** Demidovich, *Problems in Mathematical Analysis*, No. 1320.

**Statement.**

Evaluate using the Weierstrass substitution:

$$
I = \int_0^{\frac{\pi}{2}} \frac{dx}{1 + \sin x + \cos x}
$$

**Intuition.**
The integrand contains a linear combination of $\sin x$ and $\cos x$ in the denominator. The universal substitution $t = \tan(\frac{x}{2})$ rationalizes all trigonometric functions simultaneously.

**Solution.**
Set $t = \tan(\frac{x}{2})$. Then:

$$
\sin x = \frac{2t}{1+t^2}, \quad \cos x = \frac{1-t^2}{1+t^2}, \quad dx = \frac{2}{1+t^2} \, dt
$$

Transform the limits:
- When $x = 0$: $t = \tan(0) = 0$.
- When $x = \frac{\pi}{2}$: $t = \tan(\frac{\pi}{4}) = 1$.

Substitute into the integral:

$$
\begin{aligned}
I &= \int_0^1 \frac{\frac{2}{1+t^2} \, dt}{1 + \frac{2t}{1+t^2} + \frac{1-t^2}{1+t^2}} \\
&= \int_0^1 \frac{2 \, dt}{(1+t^2) + 2t + (1-t^2)} \\
&= \int_0^1 \frac{2 \, dt}{2 + 2t} = \int_0^1 \frac{dt}{1 + t}
\end{aligned}
$$

Evaluate the simplified logarithm:

$$
I = \Big[ \ln(1 + t) \Big]_0^1 = \ln(2) - \ln(1) = \ln 2
$$

$$
\boxed{\ln 2}
$$

**Key takeaway.**
The Weierstrass substitution $t = \tan(\frac{x}{2})$ is the "universal key" that maps trigonometric rational functions directly onto standard algebraic fractions.

In [12]:
print("Problem L1.6 - Weierstrass substitution, both sides")
lhs, _ = integrate.quad(lambda z: 1.0 / (1 + np.sin(z) + np.cos(z)), 0.0, np.pi / 2)
rhs, _ = integrate.quad(lambda s: 1.0 / (1 + s), 0.0, 1.0)
print("  trigonometric side:")
close(lhs, np.log(2))
print("  rationalised side:")
close(rhs, np.log(2))

Problem L1.6 - Weierstrass substitution, both sides
  trigonometric side:
  computed 0.693147180560   boxed 0.693147180560   |diff| = 1.11e-16   OK
  rationalised side:
  computed 0.693147180560   boxed 0.693147180560   |diff| = 1.11e-16   OK


True

### Problem L1.7 — Integrating the Logarithm by Parts
**Source Attribution:** Spivak, *Calculus* (4th Ed.), Chapter 18.

**Statement.**

Evaluate the indefinite integral:

$$
I = \int \ln x \, dx
$$

**Intuition.**
Even though $\ln x$ appears as a single function, integration by parts applies by setting $dv = 1 \, dx$.

**Solution.**
Let $u = \ln x \implies du = \frac{1}{x} dx$.  
Let $dv = dx \implies v = x$.  
Applying IBP ($\int u \, dv = uv - \int v \, du$):

$$
I = x \ln x - \int x \cdot \left(\frac{1}{x}\right) dx = x \ln x - \int 1 \, dx = x \ln x - x + C
$$

$$
\boxed{x \ln x - x + C}
$$

**Key takeaway.**
Integration by parts works effectively on single functions whose derivatives are simpler algebraic expressions by selecting $dv = dx$.

In [13]:
print("Problem L1.7 - differentiate x ln x - x")
F = x * sp.log(x) - x
print("  d/dx F =", sp.simplify(sp.diff(F, x)))
assert sp.simplify(sp.diff(F, x) - sp.log(x)) == 0

Problem L1.7 - differentiate x ln x - x
  d/dx F = log(x)


### Problem L1.8 — Leibniz Rule with Two Moving Boundaries
**Source Attribution:** Demidovich, *Problems in Mathematical Analysis*, No. 1450.

**Statement.**

Calculate $\frac{d}{dx} \left( \int_{\sin x}^{\cos x} e^{t^2} \, dt \right)$.

**Intuition.**
Both integration limits are differentiable functions of $x$, while the integrand depends only on $t$. We apply Leibniz's rule with variable endpoints.

**Solution.**
By the Leibniz Integral Rule:

$$
\frac{d}{dx} \int_{a(x)}^{b(x)} f(t) \, dt = f(b(x)) \cdot b'(x) - f(a(x)) \cdot a'(x)
$$

Here $f(t) = e^{t^2}$, $b(x) = \cos x \implies b'(x) = -\sin x$, and $a(x) = \sin x \implies a'(x) = \cos x$.

Substitute directly into the formula:

$$
\begin{aligned}
\frac{d}{dx} \int_{\sin x}^{\cos x} e^{t^2} \, dt &= e^{(\cos x)^2} \cdot (-\sin x) - e^{(\sin x)^2} \cdot (\cos x) \\
&= -\sin x \, e^{\cos^2 x} - \cos x \, e^{\sin^2 x}
\end{aligned}
$$

$$
\boxed{-\sin x \, e^{\cos^2 x} - \cos x \, e^{\sin^2 x}}
$$

**Key takeaway.**
Variable-boundary integrals act as composite mappings, evaluated using boundary evaluations multiplied by endpoint velocity vectors.

In [14]:
print("Problem L1.8 - Leibniz rule against a central difference, at x = 0.6")
H = lambda z: integrate.quad(lambda s: np.exp(s**2), np.sin(z), np.cos(z))[0]
x0, h = 0.6, 1e-5
formula = (-np.sin(x0) * np.exp(np.cos(x0) ** 2)
           - np.cos(x0) * np.exp(np.sin(x0) ** 2))
close((H(x0 + h) - H(x0 - h)) / (2 * h), formula, tol=1e-6)

Problem L1.8 - Leibniz rule against a central difference, at x = 0.6
  computed -2.251103356299   boxed -2.251103356392   |diff| = 9.34e-11   OK


True

### Problem L1.9 — Irreducible Quadratic and the Arctangent Form
**Source Attribution:** Stewart, *Calculus: Early Transcendentals* (8th Ed.), Chapter 7.

**Statement.**

Compute:

$$
I = \int \frac{dx}{x^2 + 4x + 13}
$$

**Intuition.**
The quadratic denominator $x^2 + 4x + 13$ has a negative discriminant ($\Delta = 16 - 52 = -36 \lt 0$) and cannot be factorized over $\mathbb{R}$. Completing the square transforms it into the standard arctangent form $\int \frac{du}{u^2 + a^2}$.

**Solution.**
Complete the square for the denominator:

$$
x^2 + 4x + 13 = (x^2 + 4x + 4) + 9 = (x + 2)^2 + 3^2
$$

Substitute $u = x + 2 \implies du = dx$:

$$
I = \int \frac{du}{u^2 + 3^2}
$$

Recall the standard arctangent integration formula $\int \frac{du}{u^2 + a^2} = \frac{1}{a} \arctan\left(\frac{u}{a}\right) + C$:

$$
I = \frac{1}{3} \arctan\left( \frac{x + 2}{3} \right) + C
$$

$$
\boxed{\frac{1}{3} \arctan\left( \frac{x + 2}{3} \right) + C}
$$

**Key takeaway.**
Completing the square maps irreducible quadratic denominators directly onto the inverse tangent antiderivative family.

In [15]:
print("Problem L1.9 - differentiate the arctangent antiderivative")
F = sp.atan((x + 2) / 3) / 3
print("  d/dx F =", sp.simplify(sp.diff(F, x)))
assert sp.simplify(sp.diff(F, x) - 1 / (x**2 + 4 * x + 13)) == 0

Problem L1.9 - differentiate the arctangent antiderivative
  d/dx F = 1/((x + 2)**2 + 9)


### Problem L1.10 — A Hidden Substitution in a Degree-Eight Denominator
**Source Attribution:** MIT Integration Bee 2021.

**Statement.**

Evaluate:

$$
I = \int_0^1 \frac{x^3}{1 + x^8} \, dx
$$

**Intuition.**
Notice that $x^8 = (x^4)^2$, and the numerator $x^3 dx$ is proportional to the differential of $x^4$. Substitution $u = x^4$ reduces the power 8 to a simple square.

**Solution.**
Let $u = x^4 \implies du = 4x^3 \, dx \implies x^3 \, dx = \frac{1}{4} du$.  
Transform limits:
- When $x = 0$: $u = 0^4 = 0$.
- When $x = 1$: $u = 1^4 = 1$.

Substitute into the integral:

$$
I = \int_0^1 \frac{\frac{1}{4} du}{1 + u^2} = \frac{1}{4} \int_0^1 \frac{du}{1 + u^2}
$$

Evaluate using $\arctan u$:

$$
I = \frac{1}{4} \Big[ \arctan u \Big]_0^1 = \frac{1}{4} \left( \arctan(1) - \arctan(0) \right) = \frac{1}{4} \cdot \frac{\pi}{4} = \frac{\pi}{16}
$$

$$
\boxed{\frac{\pi}{16}}
$$

**Key takeaway.**
Targeted substitution uncovers hidden lower-degree structures in high-power polynomials.

In [16]:
print("Problem L1.10 - int_0^1 x^3/(1+x^8) dx")
val, _ = integrate.quad(lambda z: z**3 / (1 + z**8), 0.0, 1.0)
close(val, np.pi / 16)

Problem L1.10 - int_0^1 x^3/(1+x^8) dx
  computed 0.196349540849   boxed 0.196349540849   |diff| = 2.53e-15   OK


True

### Problem L1.11 — Self-Referential Parts for $\sec^3 x$
**Source Attribution:** Stewart, *Calculus: Early Transcendentals* (8th Ed.), Chapter 7.

**Statement.**

Derive the antiderivative of $I = \int \sec^3 x \, dx$.

**Intuition.**
Split $\sec^3 x = \sec x \cdot \sec^2 x$ and use integration by parts with $dv = \sec^2 x \, dx$. The identity $\tan^2 x = \sec^2 x - 1$ leads to a self-referential integral equation for $I$.

**Solution.**
Let $u = \sec x \implies du = \sec x \tan x \, dx$.  
Let $dv = \sec^2 x \, dx \implies v = \tan x$.  
Apply IBP:

$$
I = \sec x \tan x - \int \tan x (\sec x \tan x) \, dx = \sec x \tan x - \int \sec x \tan^2 x \, dx
$$

Substitute $\tan^2 x = \sec^2 x - 1$:

$$
I = \sec x \tan x - \int \sec x (\sec^2 x - 1) \, dx = \sec x \tan x - \int \sec^3 x \, dx + \int \sec x \, dx
$$

Recognize the original integral $I = \int \sec^3 x \, dx$:

$$
I = \sec x \tan x - I + \ln \lvert \sec x + \tan x \rvert
$$

Add $I$ to both sides:

$$
2I = \sec x \tan x + \ln \lvert \sec x + \tan x \rvert \implies I = \frac{1}{2} \sec x \tan x + \frac{1}{2} \ln \lvert \sec x + \tan x \rvert + C
$$

$$
\boxed{\frac{1}{2} \sec x \tan x + \frac{1}{2} \ln \lvert \sec x + \tan x \rvert + C}
$$

**Key takeaway.**
Self-referential integration by parts resolves recursive trigonometric power integrals algebraically without infinite loops.

In [17]:
print("Problem L1.11 - differentiate the sec^3 antiderivative on (0, pi/2)")
F = (sp.sec(x) * sp.tan(x) + sp.log(sp.sec(x) + sp.tan(x))) / 2
residual = sp.simplify(sp.diff(F, x) - sp.sec(x) ** 3)
print("  d/dx F - sec^3 x =", residual)
assert residual == 0

Problem L1.11 - differentiate the sec^3 antiderivative on (0, pi/2)


  d/dx F - sec^3 x = 0


### Problem L1.12 — A Constant Function from Darboux Sums
**Source Attribution:** Spivak, *Calculus* (4th Ed.), Chapter 13.

**Statement.**

Let $f(x) = c$ be a constant function defined on $[a, b]$. Prove from first principles using Darboux upper and lower sums that $f$ is Darboux integrable on $[a, b]$ and that $\int_a^b c \, dx = c(b - a)$.

**Intuition.**
For a constant function, the supremum and infimum of $f(x)$ over any subinterval $[x_{i-1}, x_i]$ are both equal to $c$. Thus, the upper and lower Darboux sums must coincide for *every* partition, rendering the upper and lower Darboux integrals identical without taking complex limits.

**Solution.**
Let $P = \{x_0, x_1, \dots, x_n\}$ be any arbitrary partition of $[a, b]$.  
For each subinterval $[x_{i-1}, x_i]$:

$$
M_i(f, P) = \sup_{x \in [x_{i-1}, x_i]} c = c, \quad m_i(f, P) = \inf_{x \in [x_{i-1}, x_i]} c = c
$$

The Upper Darboux Sum is:

$$
U(f, P) = \sum_{i=1}^n M_i \Delta x_i = \sum_{i=1}^n c (x_i - x_{i-1}) = c \sum_{i=1}^n (x_i - x_{i-1}) = c (b - a)
$$

The Lower Darboux Sum is:

$$
L(f, P) = \sum_{i=1}^n m_i \Delta x_i = \sum_{i=1}^n c (x_i - x_{i-1}) = c (b - a)
$$

Since $U(f, P) = L(f, P) = c(b - a)$ for all partitions $P$, the upper and lower Darboux integrals are:

$$
\overline{\int_a^b} c \, dx = \inf_P U(f, P) = c(b - a), \quad \underline{\int_a^b} c \, dx = \sup_P L(f, P) = c(b - a)
$$

Because $\overline{\int_a^b} c \, dx = \underline{\int_a^b} c \, dx$, $f$ is Darboux integrable and:

$$
\int_a^b c \, dx = c(b - a)
$$

$$
\boxed{c(b - a)}
$$

**Key takeaway.**
The Darboux integral formalizes the geometric intuition that the area under a flat line of height $c$ over an interval of length $b-a$ is exactly $c(b-a)$.

In [18]:
print("Problem L1.12 - Darboux sums of a constant on a random partition")
c, a_, b_ = 2.5, -1.0, 3.0
edges = np.unique(np.concatenate(([a_, b_], rng.uniform(a_, b_, 7))))
widths = np.diff(edges)
L_sum = float((c * widths).sum())
U_sum = float((c * widths).sum())
print(f"  partition with {len(widths)} random subintervals")
print("  lower sum:")
close(L_sum, c * (b_ - a_))
print("  upper sum:")
close(U_sum, c * (b_ - a_))
print("  U - L = 0 for every partition, so the Darboux criterion holds trivially")

Problem L1.12 - Darboux sums of a constant on a random partition
  partition with 8 random subintervals
  lower sum:
  computed 10.000000000000   boxed 10.000000000000   |diff| = 0.00e+00   OK
  upper sum:
  computed 10.000000000000   boxed 10.000000000000   |diff| = 0.00e+00   OK
  U - L = 0 for every partition, so the Darboux criterion holds trivially


### Problem L1.13 — The Dirichlet Function Is Not Integrable
**Source Attribution:** Apostol, *Calculus, Volume I* (2nd Ed.), Chapter 1.

**Statement.**

Consider the Dirichlet function $f: [0, 1] \to \mathbb{R}$ defined by:

$$
f(x) = \begin{cases} 1 & \text{if } x \in \mathbb{Q} \\ 0 & \text{if } x \notin \mathbb{Q} \end{cases}
$$

Show that $f$ is NOT Riemann or Darboux integrable on $[0, 1]$.

**Intuition.**
Every subinterval of $[0, 1]$, no matter how small, contains both rational and irrational numbers due to the density of $\mathbb{Q}$ and $\mathbb{R} \setminus \mathbb{Q}$. Therefore, the supremum on any subinterval is always 1, while the infimum is always 0.

**Solution.**
Let $P = \{x_0, x_1, \dots, x_n\}$ be any partition of $[0, 1]$.  
For every subinterval $[x_{i-1}, x_i]$ with $x_{i-1} \lt x_i$:
1. By the density of rationals in $\mathbb{R}$, there exists $q \in [x_{i-1}, x_i] \cap \mathbb{Q}$, so $M_i = \sup_{x \in [x_{i-1}, x_i]} f(x) = 1$.
2. By the density of irrationals in $\mathbb{R}$, there exists $r \in [x_{i-1}, x_i] \setminus \mathbb{Q}$, so $m_i = \inf_{x \in [x_{i-1}, x_i]} f(x) = 0$.

Compute the Darboux sums for $P$:

$$
U(f, P) = \sum_{i=1}^n 1 \cdot \Delta x_i = 1 \sum_{i=1}^n \Delta x_i = 1 \cdot (1 - 0) = 1
$$

$$
L(f, P) = \sum_{i=1}^n 0 \cdot \Delta x_i = 0
$$

Taking the infimum and supremum across all partitions:

$$
\overline{\int_0^1} f(x) \, dx = \inf_P U(f, P) = 1, \quad \underline{\int_0^1} f(x) \, dx = \sup_P L(f, P) = 0
$$

Since $\overline{\int_0^1} f(x) dx = 1 \neq 0 = \underline{\int_0^1} f(x) dx$, $f$ fails the integrability condition.

$$
\boxed{\text{Not Integrable (Upper integral } 1 \ne \text{ Lower integral } 0)}
$$

**Key takeaway.**
Riemann integration cannot handle functions with dense, everywhere-discontinuous jumps. This limitation historically motivated Lebesgue's measure-theoretic integral.

In [19]:
print("Problem L1.13 - the Dirichlet gap never shrinks")
for n in (10, 1_000, 100_000):
    widths = np.full(n, 1.0 / n)
    # every subinterval meets Q and R \ Q, so M_i = 1 and m_i = 0 whatever the mesh
    U_sum = float((np.ones(n) * widths).sum())
    L_sum = float((np.zeros(n) * widths).sum())
    print(f"  n = {n:>7}: mesh = {1/n:.1e},  U = {U_sum:.3f},  L = {L_sum:.3f},  U - L = {U_sum - L_sum:.3f}")
    assert abs((U_sum - L_sum) - 1.0) < 1e-12
print("  two tagged Riemann sums of the SAME partition give 1 (rational tags) and 0")
print("  (irrational tags), so no limit exists as the mesh tends to 0")

Problem L1.13 - the Dirichlet gap never shrinks
  n =      10: mesh = 1.0e-01,  U = 1.000,  L = 0.000,  U - L = 1.000
  n =    1000: mesh = 1.0e-03,  U = 1.000,  L = 0.000,  U - L = 1.000
  n =  100000: mesh = 1.0e-05,  U = 1.000,  L = 0.000,  U - L = 1.000
  two tagged Riemann sums of the SAME partition give 1 (rational tags) and 0
  (irrational tags), so no limit exists as the mesh tends to 0


### Problem L1.14 — The $p$-Integral Convergence Test
**Source Attribution:** Stewart, *Calculus: Early Transcendentals* (8th Ed.), Chapter 7.

**Statement.**

Determine for which values of $p \in \mathbb{R}$ the improper integral $\int_1^\infty \frac{1}{x^p} \, dx$ converges, and compute its value when convergent.

**Intuition.**
As $x \to \infty$, $x^{-p}$ decays rapidly if $p \gt 1$, slowly if $0 \lt p \le 1$, and grows if $p \le 0$. Convergence depends on whether the tail area decays fast enough.

**Solution.**
By definition of improper integral:

$$
\int_1^\infty \frac{1}{x^p} \, dx = \lim_{t \to \infty} \int_1^t x^{-p} \, dx
$$

**Case 1 ($p = 1$):**

$$
\lim_{t \to \infty} \left[ \ln x \right]_1^t = \lim_{t \to \infty} (\ln t - 0) = +\infty \quad (\text{Diverges})
$$

**Case 2 ($p \neq 1$):**

$$
\lim_{t \to \infty} \left[ \frac{x^{1-p}}{1-p} \right]_1^t = \lim_{t \to \infty} \frac{t^{1-p} - 1}{1-p}
$$

- If $1 - p \gt 0 \iff p \lt 1$: $\lim_{t \to \infty} t^{1-p} = \infty$, so the integral diverges.
- If $1 - p \lt 0 \iff p \gt 1$: $\lim_{t \to \infty} t^{1-p} = 0$, so:

$$
\int_1^\infty \frac{1}{x^p} \, dx = \frac{0 - 1}{1 - p} = \frac{1}{p - 1}
$$

$$
\boxed{\text{Converges for } p \gt 1 \text{ to } \frac{1}{p - 1}; \quad \text{Diverges for } p \le 1}
$$

**Key takeaway.**
The $p$-integral test is a cornerstone criterion for convergence in analysis and continuous probability (heavy-tailed power law distributions).

In [20]:
print("Problem L1.14 - the p-integral test")
for p in (1.5, 2.0, 3.0):
    val, _ = integrate.quad(lambda z, p=p: z ** (-p), 1.0, np.inf)
    print(f"  p = {p}: converges,")
    close(val, 1.0 / (p - 1))
for p in (0.5, 1.0):
    partials = [np.log(T) if p == 1.0 else (T ** (1 - p) - 1) / (1 - p)
                for T in (1e2, 1e4, 1e8)]
    print(f"  p = {p}: partial integrals up to T = 1e2, 1e4, 1e8 -> "
          f"{np.array2string(np.array(partials), precision=2)}  (unbounded, diverges)")

Problem L1.14 - the p-integral test
  p = 1.5: converges,
  computed 2.000000000000   boxed 2.000000000000   |diff| = 0.00e+00   OK
  p = 2.0: converges,
  computed 1.000000000000   boxed 1.000000000000   |diff| = 0.00e+00   OK
  p = 3.0: converges,
  computed 0.500000000000   boxed 0.500000000000   |diff| = 0.00e+00   OK
  p = 0.5: partial integrals up to T = 1e2, 1e4, 1e8 -> [   18.   198. 19998.]  (unbounded, diverges)
  p = 1.0: partial integrals up to T = 1e2, 1e4, 1e8 -> [ 4.61  9.21 18.42]  (unbounded, diverges)


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Work Done by a Duffing Spring
**Source Attribution:** Physics / Classical Mechanics.

**Statement.**

A particle moves along the $x$-axis subject to a non-linear spring force $F(x) = -k x - \beta x^3$, where $k \gt 0$ and $\beta \gt 0$. Calculate the work done by the spring on the particle as it moves from $x = 0$ to $x = x_0$.

**Intuition.**
Work is the line integral of force over displacement: $W = \int_{0}^{x_0} F(x) \, dx$. Since the force varies non-linearly with displacement, continuous accumulation is mandatory.

**Solution.**
Set up the work integral:

$$
W = \int_0^{x_0} \left( -k x - \beta x^3 \right) \, dx
$$

Evaluate using standard polynomial integration:

$$
W = \left[ -\frac{1}{2} k x^2 - \frac{1}{4} \beta x^4 \right]_0^{x_0} = -\frac{1}{2} k x_0^2 - \frac{1}{4} \beta x_0^4
$$

The negative sign indicates that the spring force opposes positive displacement, storing potential energy $U(x_0) = \frac{1}{2} k x_0^2 + \frac{1}{4} \beta x_0^4$.

$$
\boxed{-\frac{1}{2} k x_0^2 - \frac{1}{4} \beta x_0^4}
$$

**Key takeaway.**
The definite integral seamlessly accumulates non-linear potential fields, generalizing linear Hooke's Law models to Duffing oscillator systems.

In [21]:
print("Problem L2.1 - work done by the Duffing spring")
k_s, beta_s, x0_s = sp.symbols("k beta x_0", positive=True)
W = sp.integrate(-k_s * x - beta_s * x**3, (x, 0, x0_s))
print("  W =", sp.simplify(W))
assert sp.simplify(W - (-k_s * x0_s**2 / 2 - beta_s * x0_s**4 / 4)) == 0
print("  numeric spot check with k=3, beta=0.5, x0=2:",
      float(W.subs({k_s: 3, beta_s: sp.Rational(1, 2), x0_s: 2})))

Problem L2.1 - work done by the Duffing spring
  W = x_0**2*(-beta*x_0**2 - 2*k)/4
  numeric spot check with k=3, beta=0.5, x0=2: -8.0


### Problem L2.2 — Potential of a Uniformly Charged Rod
**Source Attribution:** Physics / Electromagnetism.

**Statement.**

A thin rigid rod of length $L$ lies along the $x$-axis from $x = 0$ to $x = L$ with uniform linear charge density $\lambda = \frac{Q}{L}$. Find the electric potential $V(d)$ at a point $P$ located on the $x$-axis at $x = L + d$ (where $d \gt 0$).

**Intuition.**
Each infinitesimal segment of the rod $dx$ carries charge $dq = \lambda dx$ and acts as a point charge at distance $r = (L + d) - x$ from point $P$. Electric potential obeys continuous superposition $V = k_e \int \frac{dq}{r}$.

**Solution.**
Set up the integral for potential $V(d)$:

$$
V(d) = k_e \int_0^L \frac{\lambda \, dx}{(L + d) - x}
$$

Substitute $u = (L + d) - x \implies du = -dx$.  
When $x = 0$, $u = L + d$; when $x = L$, $u = d$:

$$
V(d) = k_e \lambda \int_d^{L+d} \frac{du}{u} = k_e \lambda \Big[ \ln u \Big]_d^{L+d} = k_e \lambda \ln\left( \frac{L + d}{d} \right)
$$

Substitute $\lambda = \frac{Q}{L}$ and $k_e = \frac{1}{4\pi \varepsilon_0}$:

$$
V(d) = \frac{Q}{4\pi \varepsilon_0 L} \ln\left( 1 + \frac{L}{d} \right)
$$

$$
\boxed{\frac{Q}{4\pi \varepsilon_0 L} \ln\left( 1 + \frac{L}{d} \right)}
$$

**Key takeaway.**
Continuous field superposition converts continuous source distributions into logarithmic potential energy fields.

In [22]:
print("Problem L2.2 - potential of a uniformly charged rod")
L_s, d_s, ke_s, lam_s = sp.symbols("L d k_e lambda", positive=True)
V = ke_s * lam_s * sp.integrate(1 / ((L_s + d_s) - x), (x, 0, L_s))
print("  V =", sp.simplify(V))
assert sp.simplify(V - ke_s * lam_s * sp.log((L_s + d_s) / d_s)) == 0
print("  with lambda = Q/L this is (Q / (4 pi eps0 L)) ln(1 + L/d)")

Problem L2.2 - potential of a uniformly charged rod


  V = log(((L + d)/d)**(k_e*lambda))
  with lambda = Q/L this is (Q / (4 pi eps0 L)) ln(1 + L/d)


### Problem L2.3 — Mean of the Exponential Distribution
**Source Attribution:** Machine Learning / Probability Theory.

**Statement.**

The continuous exponential probability density function (PDF) modeling survival time or inter-arrival time is $p(x) = \lambda e^{-\lambda x}$ for $x \ge 0$ ($\lambda \gt 0$). Compute the expected value $\mathbb{E}[X] = \int_0^\infty x \, p(x) \, dx$.

**Intuition.**
$\mathbb{E}[X]$ measures the probability-weighted center of mass of the continuous distribution across $[0, \infty)$. Integration by parts evaluates this improper expectation integral.

**Solution.**
Set up the improper integral:

$$
\mathbb{E}[X] = \int_0^\infty x \left( \lambda e^{-\lambda x} \right) \, dx = \lim_{t \to \infty} \lambda \int_0^t x e^{-\lambda x} \, dx
$$

Apply IBP: let $u = x \implies du = dx$, and $dv = e^{-\lambda x} dx \implies v = -\frac{1}{\lambda} e^{-\lambda x}$.

$$
\int x e^{-\lambda x} \, dx = -\frac{x}{\lambda} e^{-\lambda x} - \int \left( -\frac{1}{\lambda} e^{-\lambda x} \right) dx = -\frac{x}{\lambda} e^{-\lambda x} - \frac{1}{\lambda^2} e^{-\lambda x}
$$

Multiply by $\lambda$:

$$
\lambda \int x e^{-\lambda x} \, dx = -x e^{-\lambda x} - \frac{1}{\lambda} e^{-\lambda x}
$$

Evaluate limits from $0$ to $t$:

$$
\mathbb{E}[X] = \lim_{t \to \infty} \left[ -t e^{-\lambda t} - \frac{1}{\lambda} e^{-\lambda t} \right] - \left[ 0 - \frac{1}{\lambda} \right]
$$

By L'Hôpital's rule, $\lim_{t \to \infty} t e^{-\lambda t} = 0$, and $\lim_{t \to \infty} e^{-\lambda t} = 0$. Thus:

$$
\mathbb{E}[X] = 0 - 0 + \frac{1}{\lambda} = \frac{1}{\lambda}
$$

$$
\boxed{\frac{1}{\lambda}}
$$

**Key takeaway.**
Integration by parts under improper limits determines mean arrival rates for continuous stochastic processes in machine learning.

In [23]:
print("Problem L2.3 - mean of the exponential density")
for lam in (0.7, 2.0, 5.0):
    val, _ = integrate.quad(lambda z, lam=lam: z * lam * np.exp(-lam * z), 0.0, np.inf)
    print(f"  lambda = {lam}:")
    close(val, 1.0 / lam)

Problem L2.3 - mean of the exponential density
  lambda = 0.7:
  computed 1.428571428571   boxed 1.428571428571   |diff| = 0.00e+00   OK
  lambda = 2.0:
  computed 0.500000000000   boxed 0.500000000000   |diff| = 2.78e-16   OK
  lambda = 5.0:
  computed 0.200000000000   boxed 0.200000000000   |diff| = 2.50e-16   OK


### Problem L2.4 — Closed-Form Flow of a Linear Neural ODE
**Source Attribution:** Machine Learning / Neural ODEs (Chen et al., 2018).

**Statement.**

In a 1D Continuous Normalizing Flow, state dynamics follow the Neural ODE $\frac{dz}{dt} = \theta z t$ for $t \in [0, 1]$. Given initial condition $z(0) = z_0$, compute the final state $z(1)$ by analytical integration.

**Intuition.**
By FTC, $z(1) = z(0) + \int_0^1 \frac{dz}{dt} dt$. Because the velocity field is separable ($\frac{dz}{z} = \theta t dt$), exact continuous accumulation determines the non-linear transformation.

**Solution.**
Separate variables:

$$
\frac{dz}{z} = \theta t \, dt
$$

Integrate both sides from $t = 0$ ($z = z_0$) to $t = 1$ ($z = z(1)$):

$$
\int_{z_0}^{z(1)} \frac{dz}{z} = \int_0^1 \theta t \, dt
$$

Evaluate both integrals:

$$
\ln\left( \frac{z(1)}{z_0} \right) = \left[ \frac{1}{2} \theta t^2 \right]_0^1 = \frac{1}{2} \theta
$$

Exponentiate both sides:

$$
\frac{z(1)}{z_0} = e^{\frac{\theta}{2}} \implies z(1) = z_0 e^{\frac{\theta}{2}}
$$

$$
\boxed{z_0 e^{\frac{\theta}{2}}}
$$

**Key takeaway.**
Neural ODEs solve continuous feature transformation by integrating dynamic neural velocity fields along time trajectories.

In [24]:
print("Problem L2.4 - closed-form neural-ODE flow versus a numerical integrator")
theta, z0 = 1.4, 0.8
sol = integrate.solve_ivp(lambda tt, zz: theta * zz * tt, (0.0, 1.0), [z0],
                          rtol=1e-11, atol=1e-13)
close(float(sol.y[0, -1]), z0 * np.exp(theta / 2), tol=1e-8)

Problem L2.4 - closed-form neural-ODE flow versus a numerical integrator
  computed 1.611002165974   boxed 1.611002165976   |diff| = 1.93e-12   OK


True

### Problem L2.5 — Differential Entropy of a Uniform Density
**Source Attribution:** Machine Learning / Information Theory.

**Statement.**

Compute the differential entropy $H(X) = -\int_{-\infty}^\infty p(x) \ln p(x) \, dx$ for a uniform continuous distribution $X \sim U[a, b]$ with density $p(x) = \frac{1}{b - a}$ for $x \in [a, b]$.

**Intuition.**
Differential entropy measures the continuous uncertainty of a random variable. For a uniform density, $p(x)$ is constant over $[a, b]$ and zero elsewhere, making $\ln p(x)$ constant.

**Solution.**
Substitute $p(x) = \frac{1}{b - a}$ on $[a, b]$ into the entropy integral:

$$
\begin{aligned}
H(X) &= -\int_a^b \frac{1}{b - a} \ln\left( \frac{1}{b - a} \right) \, dx \\
&= -\ln\left( \frac{1}{b - a} \right) \cdot \frac{1}{b - a} \int_a^b 1 \, dx \\
&= \ln(b - a) \cdot \frac{1}{b - a} \cdot (b - a) = \ln(b - a)
\end{aligned}
$$

$$
\boxed{\ln(b - a)}
$$

**Key takeaway.**
The continuous entropy of a uniform distribution scales logarithmically with the domain width $b - a$.

In [25]:
print("Problem L2.5 - differential entropy of U[a, b]")
for a_, b_ in ((0.0, 1.0), (-2.0, 3.5), (1.0, 1.25)):
    p = 1.0 / (b_ - a_)
    val, _ = integrate.quad(lambda z, p=p: -p * np.log(p), a_, b_)
    print(f"  U[{a_}, {b_}]:")
    close(val, np.log(b_ - a_))

Problem L2.5 - differential entropy of U[a, b]
  U[0.0, 1.0]:
  computed -0.000000000000   boxed 0.000000000000   |diff| = 0.00e+00   OK
  U[-2.0, 3.5]:
  computed 1.704748092238   boxed 1.704748092238   |diff| = 0.00e+00   OK
  U[1.0, 1.25]:
  computed -1.386294361120   boxed -1.386294361120   |diff| = 0.00e+00   OK


### Problem L2.6 — The Gaussian Integral by Polar Coordinates
**Source Attribution:** Physics & AI / Gaussian Integrals.

**Statement.**

Compute the Gaussian integral $I = \int_{-\infty}^\infty e^{-x^2} \, dx$ by evaluating the double integral $I^2 = \iint_{\mathbb{R}^2} e^{-(x^2+y^2)} \, dx \, dy$ in polar coordinates.

**Intuition.**
$e^{-x^2}$ has no elementary antiderivative. Squaring $I$ turns the 1D problem into a 2D circularly symmetric surface over $\mathbb{R}^2$, where polar coordinates $(r, \theta)$ introduce the Jacobian factor $r \, dr \, d\theta$.

**Solution.**
Express $I^2$ as a double integral:

$$
I^2 = \left( \int_{-\infty}^\infty e^{-x^2} dx \right) \left( \int_{-\infty}^\infty e^{-y^2} dy \right) = \int_{-\infty}^\infty \int_{-\infty}^\infty e^{-(x^2+y^2)} \, dx \, dy
$$

Convert to polar coordinates: $x = r \cos \theta$, $y = r \sin \theta$, $x^2 + y^2 = r^2$, $dx \, dy = r \, dr \, d\theta$:

$$
I^2 = \int_0^{2\pi} d\theta \int_0^\infty e^{-r^2} r \, dr = 2\pi \int_0^\infty r e^{-r^2} \, dr
$$

Substitute $u = r^2 \implies du = 2r \, dr \implies r \, dr = \frac{1}{2} du$:

$$
I^2 = 2\pi \int_0^\infty e^{-u} \left( \frac{1}{2} du \right) = \pi \Big[ -e^{-u} \Big]_0^\infty = \pi (0 - (-1)) = \pi
$$

Take the square root (since $e^{-x^2} \gt 0 \implies I \gt 0$):

$$
I = \sqrt{\pi}
$$

$$
\boxed{\sqrt{\pi}}
$$

**Key takeaway.**
Multi-dimensional coordinate transformations unlock closed-form evaluations for non-elementary 1D integrals essential to Gaussian probabilities and machine learning.

In [26]:
print("Problem L2.6 - the Gaussian integral")
val, _ = integrate.quad(lambda z: np.exp(-z**2), -np.inf, np.inf)
close(val, np.sqrt(np.pi))
polar, _ = integrate.quad(lambda r: 2 * np.pi * r * np.exp(-r**2), 0.0, np.inf)
print("  polar double integral I^2 =", f"{polar:.12f}", " pi =", f"{np.pi:.12f}")
close(polar, np.pi)

Problem L2.6 - the Gaussian integral
  computed 1.772453850906   boxed 1.772453850906   |diff| = 0.00e+00   OK
  polar double integral I^2 = 3.141592653590  pi = 3.141592653590
  computed 3.141592653590   boxed 3.141592653590   |diff| = 4.44e-16   OK


True

### Problem L2.7 — Expected $L_1$ Risk Under a Laplace Likelihood
**Source Attribution:** Machine Learning / Statistical Learning Theory.

**Statement.**

Suppose target data obeys the Laplace probability distribution $p(y \mid x) = \frac{1}{2} e^{-\lvert y - \theta x \rvert}$. Calculate the expected loss $R(\theta) = \int_{-\infty}^\infty \lvert y - \theta x \rvert p(y \mid x) \, dy$.

**Intuition.**
The integral measures expected absolute error ($L_1$ loss). Splitting the domain at the symmetry point $y = \theta x$ eliminates absolute value signs.

**Solution.**
Substitute $p(y \mid x)$:

$$
R(\theta) = \int_{-\infty}^\infty \lvert y - \theta x \rvert \left( \frac{1}{2} e^{-\lvert y - \theta x \rvert} \right) dy
$$

Let $u = y - \theta x \implies du = dy$:

$$
R(\theta) = \frac{1}{2} \int_{-\infty}^\infty \lvert u \rvert e^{-\lvert u \rvert} \, du
$$

By even symmetry of $\lvert u \rvert e^{-\lvert u \rvert}$:

$$
R(\theta) = \frac{1}{2} \cdot 2 \int_0^\infty u e^{-u} \, du = \int_0^\infty u e^{-u} \, du
$$

Evaluate using IBP ($\int_0^\infty u e^{-u} du = 1$):

$$
R(\theta) = 1
$$

$$
\boxed{1}
$$

**Key takeaway.**
Symmetry and variable shifting decouple model parameters from expectation integrals in maximum likelihood estimation.

In [27]:
print("Problem L2.7 - expected L1 risk under a Laplace likelihood")
for theta_x in (0.0, -3.2, 7.5):
    val, _ = integrate.quad(lambda yy, m=theta_x: abs(yy - m) * 0.5 * np.exp(-abs(yy - m)),
                            -np.inf, np.inf)
    print(f"  theta*x = {theta_x}:")
    close(val, 1.0, tol=1e-8)
print("  the risk is 1 regardless of theta: the location parameter drops out")

Problem L2.7 - expected L1 risk under a Laplace likelihood
  theta*x = 0.0:
  computed 1.000000000000   boxed 1.000000000000   |diff| = 2.22e-16   OK
  theta*x = -3.2:
  computed 1.000000000046   boxed 1.000000000000   |diff| = 4.62e-11   OK
  theta*x = 7.5:
  computed 0.999999999959   boxed 1.000000000000   |diff| = 4.10e-11   OK
  the risk is 1 regardless of theta: the location parameter drops out


### Problem L2.8 — Simpson's Rule Error Bound
**Source Attribution:** Numerical Analysis / Quadrature Bounds.

**Statement.**

Calculate the exact upper bound for the error when approximating $\int_0^1 e^{-x} \, dx$ using Simpson's $\frac{1}{3}$ Rule with $n = 2$ subintervals.

**Intuition.**
The error bound for Simpson's Rule is given by $\lvert E_S \rvert \le \frac{(b-a)^5}{180 n^4} \max_{\xi \in [a, b]} \lvert f^{(4)}(\xi) \rvert$. High derivative bounds quantify the maximum truncation error.

**Solution.**
1. Identify parameters: $a = 0, b = 1, n = 2 \implies b - a = 1$.
2. Compute the 4th derivative of $f(x) = e^{-x}$:

$$
f'(x) = -e^{-x}, \quad f''(x) = e^{-x}, \quad f'''(x) = -e^{-x}, \quad f^{(4)}(x) = e^{-x}
$$

3. Find $\max_{\xi \in [0, 1]} \lvert f^{(4)}(\xi) \rvert$:
   Since $e^{-x}$ is strictly decreasing on $[0, 1]$, its maximum occurs at $\xi = 0$:

$$
K_4 = \max_{\xi \in [0, 1]} e^{-\xi} = e^0 = 1
$$

4. Compute the error bound:

$$
\lvert E_S \rvert \le \frac{(1 - 0)^5}{180 \cdot 2^4} \cdot 1 = \frac{1}{180 \cdot 16} = \frac{1}{2880}
$$

$$
\boxed{\frac{1}{2880} \approx 0.000347}
$$

**Key takeaway.**
Fourth-order derivative bounds guarantee high accuracy ($O(h^4)$ convergence) for smooth integrands evaluated via Simpson's rule.

In [28]:
print("Problem L2.8 - Simpson error bound versus the realised error")
a_, b_, n = 0.0, 1.0, 2
xs = np.linspace(a_, b_, n + 1)
gg = np.exp(-xs)
est = (b_ - a_) / (3 * n) * (gg[0] + 4 * gg[1] + gg[2])
exact = 1 - np.exp(-1.0)
bound = (b_ - a_) ** 5 / (180 * n**4) * 1.0   # max of the 4th derivative on [0,1]
print(f"  Simpson estimate = {est:.10f},  exact = {exact:.10f}")
print(f"  realised error   = {abs(est - exact):.3e}")
print("  bound (b-a)^5 / (180 n^4) * K4 = 1/2880:")
close(bound, 1.0 / 2880, tol=1e-12)
assert abs(est - exact) <= bound

Problem L2.8 - Simpson error bound versus the realised error
  Simpson estimate = 0.6323336800,  exact = 0.6321205588
  realised error   = 2.131e-04
  bound (b-a)^5 / (180 n^4) * K4 = 1/2880:
  computed 0.000347222222   boxed 0.000347222222   |diff| = 0.00e+00   OK


### Problem L2.9 — Partition Function of a Gaussian Energy Model
**Source Attribution:** Machine Learning / Softmax & VAEs.

**Statement.**

Compute the partition function normalization constant $Z = \int_{-\infty}^\infty e^{-\frac{x^2}{2\sigma^2}} \, dx$ for a Gaussian energy model.

**Intuition.**
Using substitution, transform $e^{-\frac{x^2}{2\sigma^2}}$ into the standard Gaussian integrand $e^{-u^2}$ whose integral is $\sqrt{\pi}$.

**Solution.**
Let $u = \frac{x}{\sqrt{2}\sigma} \implies dx = \sqrt{2}\sigma \, du$.  
Substitute into the integral:

$$
Z = \int_{-\infty}^\infty e^{-u^2} \left( \sqrt{2}\sigma \, du \right) = \sqrt{2}\sigma \int_{-\infty}^\infty e^{-u^2} \, du
$$

Using the standard Gaussian result $\int_{-\infty}^\infty e^{-u^2} du = \sqrt{\pi}$:

$$
Z = \sqrt{2}\sigma \cdot \sqrt{\pi} = \sigma \sqrt{2\pi}
$$

$$
\boxed{\sigma \sqrt{2\pi}}
$$

**Key takeaway.**
Linear rescaling of probability density domains determines the exact scaling factors of continuous partition functions.

In [29]:
print("Problem L2.9 - Gaussian partition function")
for sigma in (0.4, 1.0, 3.7):
    val, _ = integrate.quad(lambda z, s=sigma: np.exp(-z**2 / (2 * s**2)), -np.inf, np.inf)
    print(f"  sigma = {sigma}:")
    close(val, sigma * np.sqrt(2 * np.pi))

Problem L2.9 - Gaussian partition function
  sigma = 0.4:
  computed 1.002651309852   boxed 1.002651309852   |diff| = 2.22e-16   OK
  sigma = 1.0:
  computed 2.506628274631   boxed 2.506628274631   |diff| = 8.88e-16   OK
  sigma = 3.7:
  computed 9.274524616135   boxed 9.274524616135   |diff| = 0.00e+00   OK


### Problem L2.10 — Centre of Mass of a Semicircular Plate
**Source Attribution:** Physics / Mechanics.

**Statement.**

Find the $y$-coordinate of the center of mass $\bar{y}$ for a thin uniform semi-circular plate of radius $R$ defined by $x^2 + y^2 \le R^2$ with $y \ge 0$.

**Intuition.**
By symmetry across the $y$-axis, $\bar{x} = 0$. $\bar{y} = \frac{\iint y \, dA}{\iint dA}$, where $\iint dA = \frac{\pi R^2}{2}$ is the area of the semi-circle.

**Solution.**
Use polar coordinates ($x = r \cos \theta, y = r \sin \theta, dA = r \, dr \, d\theta$) with $r \in [0, R]$ and $\theta \in [0, \pi]$:

$$
\text{Numerator} = \iint y \, dA = \int_0^\pi \int_0^R (r \sin \theta) r \, dr \, d\theta = \left( \int_0^\pi \sin \theta \, d\theta \right) \left( \int_0^R r^2 \, dr \right)
$$

Evaluate the single integrals:
- $\int_0^\pi \sin \theta \, d\theta = \Big[ -\cos \theta \Big]_0^\pi = -(-1) - (-1) = 2$.
- $\int_0^R r^2 \, dr = \frac{R^3}{3}$.

Thus $\text{Numerator} = 2 \cdot \frac{R^3}{3} = \frac{2R^3}{3}$.  
Denominator (Area) $= \frac{\pi R^2}{2}$.  
Compute $\bar{y}$:

$$
\bar{y} = \frac{\frac{2R^3}{3}}{\frac{\pi R^2}{2}} = \frac{2R^3}{3} \cdot \frac{2}{\pi R^2} = \frac{4R}{3\pi}
$$

$$
\boxed{\frac{4R}{3\pi}}
$$

**Key takeaway.**
Double integrals in polar coordinates locate geometric centroids by accumulating moment arms against surface area elements.

In [30]:
print("Problem L2.10 - centroid height of a semicircular plate")
R = 2.3
num, _ = integrate.dblquad(lambda r, th: (r * np.sin(th)) * r, 0.0, np.pi, 0.0, R)
den = np.pi * R**2 / 2
print(f"  numerator = {num:.10f}  (hand 2R^3/3 = {2 * R**3 / 3:.10f})")
close(num / den, 4 * R / (3 * np.pi))

Problem L2.10 - centroid height of a semicircular plate
  numerator = 8.1113333333  (hand 2R^3/3 = 8.1113333333)
  computed 0.976150317630   boxed 0.976150317630   |diff| = 0.00e+00   OK


True

### Problem L2.11 — AUC as a Change-of-Variable Integral
**Source Attribution:** Machine Learning / Classification Metrics.

**Statement.**

Calculate the Area Under the ROC Curve (AUC) for a score classifier where negative instances follow $p_0(x) = e^{-x}$ and positive instances follow $p_1(x) = 2e^{-2x}$ for $x \ge 0$. Note that $\operatorname{FPR}(t) = \int_t^\infty p_0(x) dx = e^{-t}$ and $\operatorname{TPR}(t) = \int_t^\infty p_1(x) dx = e^{-2t}$.

**Intuition.**
The AUC is defined as the parametric integral $\operatorname{AUC} = \int_0^1 \operatorname{TPR} \, d(\operatorname{FPR})$. As threshold $t$ moves from $\infty$ to $0$, $\operatorname{FPR}$ moves from $0$ to $1$.

**Solution.**
Let $u = \operatorname{FPR}(t) = e^{-t} \implies t = -\ln u$.  
Substitute $t$ into $\operatorname{TPR}(t)$:

$$
\operatorname{TPR}(u) = e^{-2(-\ln u)} = e^{\ln(u^2)} = u^2
$$

The AUC integral with respect to $u \in [0, 1]$ becomes:

$$
\operatorname{AUC} = \int_0^1 u^2 \, du = \left[ \frac{u^3}{3} \right]_0^1 = \frac{1}{3}
$$

$$
\boxed{\frac{1}{3}}
$$

**Key takeaway.**
AUC metrics represent standard change-of-variable integrals mapping decision thresholds onto false positive rate coordinates.

In [31]:
print("Problem L2.11 - AUC by change of variables and by Monte Carlo")
val, _ = integrate.quad(lambda uu: uu**2, 0.0, 1.0)
print("  int_0^1 u^2 du:")
close(val, 1 / 3)
neg = rng.exponential(scale=1.0, size=2_000_000)          # p0(x) = e^{-x}
pos = rng.exponential(scale=0.5, size=2_000_000)          # p1(x) = 2 e^{-2x}
mc = float((pos > neg).mean())
print(f"  Monte Carlo P(score_pos > score_neg) = {mc:.5f}   (boxed 1/3 = {1/3:.5f})")
assert abs(mc - 1 / 3) < 2e-3

Problem L2.11 - AUC by change of variables and by Monte Carlo
  int_0^1 u^2 du:
  computed 0.333333333333   boxed 0.333333333333   |diff| = 5.55e-17   OK
  Monte Carlo P(score_pos > score_neg) = 0.33315   (boxed 1/3 = 0.33333)


## L3 — Challenge Proofs

### Problem L3.1 — King's Property on a Symmetric Trigonometric Ratio
**Source Attribution:** Putnam 1980 A1 / Symmetrical Integrals.

**Statement.**

Evaluate:

$$
I = \int_0^{\frac{\pi}{2}} \frac{\sin^\alpha x}{\sin^\alpha x + \cos^\alpha x} \, dx \quad (\alpha \in \mathbb{R})
$$

**Intuition.**
Apply the "King's Property" integral substitution $x \to a + b - x$. For limits $[0, \frac{\pi}{2}]$, $x \to \frac{\pi}{2} - x$ swaps $\sin x$ and $\cos x$.

**Solution.**
Let $x = \frac{\pi}{2} - u \implies dx = -du$.  
Limits: when $x = 0$, $u = \frac{\pi}{2}$; when $x = \frac{\pi}{2}$, $u = 0$.

$$
I = \int_{\frac{\pi}{2}}^0 \frac{\sin^\alpha(\frac{\pi}{2}-u)}{\sin^\alpha(\frac{\pi}{2}-u) + \cos^\alpha(\frac{\pi}{2}-u)} (-du) = \int_0^{\frac{\pi}{2}} \frac{\cos^\alpha u}{\cos^\alpha u + \sin^\alpha u} \, du
$$

Add the original expression for $I$ and this transformed expression:

$$
2I = \int_0^{\frac{\pi}{2}} \frac{\sin^\alpha x}{\sin^\alpha x + \cos^\alpha x} \, dx + \int_0^{\frac{\pi}{2}} \frac{\cos^\alpha x}{\sin^\alpha x + \cos^\alpha x} \, dx = \int_0^{\frac{\pi}{2}} \frac{\sin^\alpha x + \cos^\alpha x}{\sin^\alpha x + \cos^\alpha x} \, dx
$$

Simplify the integrand:

$$
2I = \int_0^{\frac{\pi}{2}} 1 \, dx = \frac{\pi}{2} \implies I = \frac{\pi}{4}
$$

$$
\boxed{\frac{\pi}{4}}
$$

**Key takeaway.**
King's Property ($\int_a^b f(x) dx = \int_a^b f(a+b-x) dx$) exploits interval reflection symmetry to cancel non-trivial integrands completely.

In [32]:
print("Problem L3.1 - King's property makes the answer independent of alpha")
for alpha in (0.0, 0.5, 1.0, 2.0, 3.0, 7.0):
    f_a = lambda z, a=alpha: np.sin(z) ** a / (np.sin(z) ** a + np.cos(z) ** a)
    val, _ = integrate.quad(f_a, 0.0, np.pi / 2)
    print(f"  alpha = {alpha}:")
    close(val, np.pi / 4, tol=1e-7)

Problem L3.1 - King's property makes the answer independent of alpha
  alpha = 0.0:
  computed 0.785398163397   boxed 0.785398163397   |diff| = 0.00e+00   OK
  alpha = 0.5:
  computed 0.785398163397   boxed 0.785398163397   |diff| = 0.00e+00   OK
  alpha = 1.0:
  computed 0.785398163397   boxed 0.785398163397   |diff| = 0.00e+00   OK
  alpha = 2.0:
  computed 0.785398163397   boxed 0.785398163397   |diff| = 0.00e+00   OK
  alpha = 3.0:
  computed 0.785398163397   boxed 0.785398163397   |diff| = 1.11e-16   OK
  alpha = 7.0:
  computed 0.785398163397   boxed 0.785398163397   |diff| = 0.00e+00   OK


### Problem L3.2 — Feynman Differentiation Removes a Log Denominator
**Source Attribution:** Demidovich No. 2244 / Feynman's Technique.

**Statement.**

Evaluate for $a \gt -1$:

$$
I(a) = \int_0^1 \frac{x^a - 1}{\ln x} \, dx
$$

**Intuition.**
The denominator $\ln x$ prevents direct integration. Differentiating $I(a)$ with respect to parameter $a$ under the integral sign (Leibniz Rule) eliminates $\ln x$ via $\frac{d}{da}(x^a) = x^a \ln x$.

**Solution.**
Differentiate $I(a)$ with respect to $a$:

$$
I'(a) = \frac{d}{da} \int_0^1 \frac{x^a - 1}{\ln x} \, dx = \int_0^1 \frac{\frac{\partial}{\partial a}(x^a - 1)}{\ln x} \, dx = \int_0^1 \frac{x^a \ln x}{\ln x} \, dx = \int_0^1 x^a \, dx
$$

Evaluate the simplified continuous integral:

$$
I'(a) = \left[ \frac{x^{a+1}}{a+1} \right]_0^1 = \frac{1}{a+1}
$$

Integrate $I'(a)$ back with respect to $a$:

$$
I(a) = \int \frac{da}{a+1} = \ln(a+1) + C
$$

Determine constant $C$ using baseline parameter $a = 0$:

$$
I(0) = \int_0^1 \frac{x^0 - 1}{\ln x} \, dx = 0 \implies \ln(0 + 1) + C = 0 \implies C = 0
$$

Thus:

$$
I(a) = \ln(a + 1)
$$

$$
\boxed{\ln(a + 1)}
$$

**Key takeaway.**
Feynman's technique of differentiation under the integral sign removes stubborn log-denominators by introducing parametric differential relationships.

In [33]:
print("Problem L3.2 - I(a) = ln(a+1) by Feynman differentiation")
for a_ in (-0.5, 0.0, 0.5, 2.0, 5.0):
    val, _ = integrate.quad(lambda z, a=a_: (z**a - 1) / np.log(z), 0.0, 1.0)
    print(f"  a = {a_}:")
    close(val, np.log(a_ + 1), tol=1e-8)

Problem L3.2 - I(a) = ln(a+1) by Feynman differentiation
  a = -0.5:
  computed -0.693147180474   boxed -0.693147180560   |diff| = 8.61e-11   OK
  a = 0.0:
  computed -0.000000000000   boxed 0.000000000000   |diff| = 0.00e+00   OK
  a = 0.5:
  computed 0.405465108108   boxed 0.405465108108   |diff| = 2.84e-13   OK
  a = 2.0:
  computed 1.098612288668   boxed 1.098612288668   |diff| = 1.89e-13   OK
  a = 5.0:
  computed 1.791759469225   boxed 1.791759469228   |diff| = 3.16e-12   OK


### Problem L3.3 — Frullani's Integral by Fubini
**Source Attribution:** Cambridge Mathematical Tripos Part IA.

**Statement.**

Evaluate Frullani's integral for $a, b \gt 0$:

$$
I = \int_0^\infty \frac{e^{-ax} - e^{-bx}}{x} \, dx
$$

**Intuition.**
Recognize that $\frac{e^{-ax} - e^{-bx}}{x} = \int_a^b e^{-ty} \, dt$ evaluated at $y = x$. Exchanging integration order via Fubini's Theorem converts the expression into a double integral.

**Solution.**
Express integrand as a parameter integral:

$$
\frac{e^{-ax} - e^{-bx}}{x} = \int_a^b e^{-tx} \, dt
$$

Substitute into $I$:

$$
I = \int_0^\infty \left( \int_a^b e^{-tx} \, dt \right) dx
$$

By Fubini's Theorem, switch integration order:

$$
I = \int_a^b \left( \int_0^\infty e^{-tx} \, dx \right) dt
$$

Evaluate the inner integral with respect to $x$:

$$
\int_0^\infty e^{-tx} \, dx = \left[ -\frac{1}{t} e^{-tx} \right]_0^\infty = \frac{1}{t}
$$

Evaluate the outer integral with respect to $t$:

$$
I = \int_a^b \frac{1}{t} \, dt = \Big[ \ln t \Big]_a^b = \ln b - \ln a = \ln\left(\frac{b}{a}\right)
$$

$$
\boxed{\ln\left(\frac{b}{a}\right)}
$$

**Key takeaway.**
Frullani integrals convert ratio-of-difference limits into clean double integrals over separable product domains.

In [34]:
print("Problem L3.3 - Frullani's integral")
for a_, b_ in ((1.0, 3.0), (0.5, 2.0), (2.0, 0.25)):
    val, _ = integrate.quad(lambda z, a=a_, b=b_: (np.exp(-a * z) - np.exp(-b * z)) / z,
                            0.0, np.inf, limit=400)
    print(f"  a = {a_}, b = {b_}:")
    close(val, np.log(b_ / a_), tol=1e-7)

Problem L3.3 - Frullani's integral
  a = 1.0, b = 3.0:
  computed 1.098612288668   boxed 1.098612288668   |diff| = 2.11e-14   OK
  a = 0.5, b = 2.0:
  computed 1.386294361120   boxed 1.386294361120   |diff| = 1.27e-14   OK
  a = 2.0, b = 0.25:
  computed -2.079441541680   boxed -2.079441541680   |diff| = 1.02e-14   OK


### Problem L3.4 — Reflection Plus Substitution on $x\sin x/(1+\cos^2 x)$
**Source Attribution:** MIT Integration Bee 2022 Final.

**Statement.**

Evaluate:

$$
I = \int_0^\pi \frac{x \sin x}{1 + \cos^2 x} \, dx
$$

**Intuition.**
The factor $x$ in the numerator prevents direct $u$-substitution. Applying King's Property ($x \to \pi - x$) eliminates $x$ entirely.

**Solution.**
Apply substitution $x = \pi - u \implies dx = -du$:

$$
I = \int_\pi^0 \frac{(\pi - u) \sin(\pi - u)}{1 + \cos^2(\pi - u)} (-du) = \int_0^\pi \frac{(\pi - u) \sin u}{1 + \cos^2 u} \, du
$$

Split the integrand:

$$
I = \pi \int_0^\pi \frac{\sin u}{1 + \cos^2 u} \, du - \int_0^\pi \frac{u \sin u}{1 + \cos^2 u} \, du = \pi \int_0^\pi \frac{\sin u}{1 + \cos^2 u} \, du - I
$$

Add $I$ to both sides:

$$
2I = \pi \int_0^\pi \frac{\sin u}{1 + \cos^2 u} \, du \implies I = \frac{\pi}{2} \int_0^\pi \frac{\sin u}{1 + \cos^2 u} \, du
$$

Now substitute $y = \cos u \implies dy = -\sin u \, du$.  
Limits: when $u = 0, y = 1$; when $u = \pi, y = -1$:

$$
I = \frac{\pi}{2} \int_1^{-1} \frac{-dy}{1 + y^2} = \frac{\pi}{2} \int_{-1}^1 \frac{dy}{1 + y^2} = \frac{\pi}{2} \Big[ \arctan y \Big]_{-1}^1 = \frac{\pi}{2} \left( \frac{\pi}{4} - \left(-\frac{\pi}{4}\right) \right) = \frac{\pi^2}{4}
$$

$$
\boxed{\frac{\pi^2}{4}}
$$

**Key takeaway.**
Combining endpoint reflection symmetry with algebraic substitution removes polynomial multiplier factors from trigonometric fractions.

In [35]:
print("Problem L3.4 - int_0^pi x sin x / (1 + cos^2 x) dx")
val, _ = integrate.quad(lambda z: z * np.sin(z) / (1 + np.cos(z) ** 2), 0.0, np.pi)
close(val, np.pi**2 / 4)

Problem L3.4 - int_0^pi x sin x / (1 + cos^2 x) dx
  computed 2.467401100272   boxed 2.467401100272   |diff| = 0.00e+00   OK


True

### Problem L3.5 — A Polynomial Kernel Concentrating at an Endpoint
**Source Attribution:** Pólya & Szegő, *Problems and Theorems in Analysis I*.

**Statement.**

Let $f: [0, 1] \to \mathbb{R}$ be continuous. Compute the asymptotic limit:

$$
L = \lim_{n \to \infty} (n + 1) \int_0^1 x^n f(x) \, dx
$$

**Intuition.**
As $n \to \infty$, the kernel $x^n$ concentrates almost all its weight near $x = 1$. The limit behaves like a Dirac delta mass focused at $x = 1$.

**Solution.**
Integrate by parts: let $u = f(x) \implies du = f'(x) dx$ (assuming $f$ is $C^1$; the result holds for continuous $f$ by approximation).  
Let $dv = (n+1)x^n dx \implies v = x^{n+1}$.

$$
(n+1) \int_0^1 x^n f(x) \, dx = \Big[ x^{n+1} f(x) \Big]_0^1 - \int_0^1 x^{n+1} f'(x) \, dx = f(1) - \int_0^1 x^{n+1} f'(x) \, dx
$$

Now bound the remaining integral:

$$
\left\vert \int_0^1 x^{n+1} f'(x) \, dx \right\vert \le \max_{x \in [0,1]} \lvert f'(x) \rvert \int_0^1 x^{n+1} \, dx = M \cdot \frac{1}{n+2}
$$

Taking $n \to \infty$:

$$
\lim_{n \to \infty} \left\vert \int_0^1 x^{n+1} f'(x) \, dx \right\vert \le \lim_{n \to \infty} \frac{M}{n+2} = 0
$$

Therefore:

$$
L = f(1) - 0 = f(1)
$$

$$
\boxed{f(1)}
$$

**Key takeaway.**
High-power polynomial kernels $x^n$ condense integral measures into boundary point evaluation operators as $n \to \infty$.

In [36]:
print("Problem L3.5 - (n+1) int_0^1 x^n f(x) dx concentrates at x = 1")
f_test = lambda z: np.cos(z) + z**2
for n in (5, 50, 500, 5000):
    val, _ = integrate.quad(lambda z, n=n: (n + 1) * z**n * f_test(z), 0.0, 1.0)
    print(f"  n = {n:>5}: value = {val:.8f}   f(1) = {f_test(1.0):.8f}   "
          f"|diff| = {abs(val - f_test(1.0)):.2e}")
close(val, f_test(1.0), tol=2e-3)

Problem L3.5 - (n+1) int_0^1 x^n f(x) dx concentrates at x = 1
  n =     5: value = 1.39931608   f(1) = 1.54030231   |diff| = 1.41e-01
  n =    50: value = 1.51854696   f(1) = 1.54030231   |diff| = 2.18e-02
  n =   500: value = 1.53800025   f(1) = 1.54030231   |diff| = 2.30e-03
  n =  5000: value = 1.54007075   f(1) = 1.54030231   |diff| = 2.32e-04
  computed 1.540070751033   boxed 1.540302305868   |diff| = 2.32e-04   OK


True

### Problem L3.6 — Wallis Reduction Formula
**Source Attribution:** Cambridge Mathematical Tripos Part IA.

**Statement.**

Derive the Wallis reduction formula for $I_n = \int_0^{\frac{\pi}{2}} \sin^n x \, dx$, and use it to compute $I_4$ and $I_5$.

**Intuition.**
Split $\sin^n x = \sin^{n-1} x \cdot \sin x$ and integrate by parts to set up a linear recursive relation between $I_n$ and $I_{n-2}$.

**Solution.**
Let $u = \sin^{n-1} x \implies du = (n-1) \sin^{n-2} x \cos x \, dx$.  
Let $dv = \sin x \, dx \implies v = -\cos x$.  
Apply IBP:

$$
I_n = \Big[ -\sin^{n-1} x \cos x \Big]_0^{\frac{\pi}{2}} + (n-1) \int_0^{\frac{\pi}{2}} \sin^{n-2} x \cos^2 x \, dx
$$

The boundary term evaluates to $0$ since $\sin(0) = 0$ and $\cos(\frac{\pi}{2}) = 0$.  
Substitute $\cos^2 x = 1 - \sin^2 x$:

$$
I_n = (n-1) \int_0^{\frac{\pi}{2}} \sin^{n-2} x (1 - \sin^2 x) \, dx = (n-1) I_{n-2} - (n-1) I_n
$$

Rearrange terms:

$$
n I_n = (n-1) I_{n-2} \implies I_n = \frac{n-1}{n} I_{n-2}
$$

Compute base cases:
- $I_0 = \int_0^{\frac{\pi}{2}} 1 \, dx = \frac{\pi}{2}$.
- $I_1 = \int_0^{\frac{\pi}{2}} \sin x \, dx = \Big[ -\cos x \Big]_0^{\frac{\pi}{2}} = 1$.

Compute $I_4$ and $I_5$:

$$
I_4 = \frac{3}{4} I_2 = \frac{3}{4} \left( \frac{1}{2} I_0 \right) = \frac{3}{4} \cdot \frac{1}{2} \cdot \frac{\pi}{2} = \frac{3\pi}{16}
$$

$$
I_5 = \frac{4}{5} I_3 = \frac{4}{5} \left( \frac{2}{3} I_1 \right) = \frac{4}{5} \cdot \frac{2}{3} \cdot 1 = \frac{8}{15}
$$

$$
\boxed{I_4 = \frac{3\pi}{16}, \quad I_5 = \frac{8}{15}}
$$

**Key takeaway.**
Wallis reduction formulas convert trigonometric power integrals into simple rational products of double factorials.

In [37]:
print("Problem L3.6 - Wallis reduction")
I = [np.pi / 2, 1.0]
for n in range(2, 6):
    I.append((n - 1) / n * I[n - 2])
for n in (4, 5):
    val, _ = integrate.quad(lambda z, n=n: np.sin(z) ** n, 0.0, np.pi / 2)
    print(f"  I_{n}: recursion gives {I[n]:.12f}, quadrature gives {val:.12f}")
    close(I[n], val)
close(I[4], 3 * np.pi / 16)
close(I[5], 8 / 15)

Problem L3.6 - Wallis reduction
  I_4: recursion gives 0.589048622548, quadrature gives 0.589048622548
  computed 0.589048622548   boxed 0.589048622548   |diff| = 0.00e+00   OK
  I_5: recursion gives 0.533333333333, quadrature gives 0.533333333333
  computed 0.533333333333   boxed 0.533333333333   |diff| = 0.00e+00   OK
  computed 0.589048622548   boxed 0.589048622548   |diff| = 0.00e+00   OK
  computed 0.533333333333   boxed 0.533333333333   |diff| = 0.00e+00   OK


True

### Problem L3.7 — The Dirichlet Integral by Exponential Regularisation
**Source Attribution:** Demidovich No. 2280 / Feynman Technique.

**Statement.**

Evaluate the conditionally convergent Dirichlet integral:

$$
I = \int_0^\infty \frac{\sin x}{x} \, dx
$$

**Intuition.**
Introduce a regularizing exponential parameter $\alpha \ge 0$: $I(\alpha) = \int_0^\infty e^{-\alpha x} \frac{\sin x}{x} \, dx$. Differentiating with respect to $\alpha$ under the integral sign eliminates $x$ in the denominator.

**Solution.**
1. Define $I(\alpha) = \int_0^\infty e^{-\alpha x} \frac{\sin x}{x} \, dx$. We seek $I(0) = I$.
2. Differentiate $I(\alpha)$ with respect to $\alpha$:

$$
I'(\alpha) = \int_0^\infty \frac{\partial}{\partial \alpha} \left( e^{-\alpha x} \frac{\sin x}{x} \right) dx = \int_0^\infty -x e^{-\alpha x} \frac{\sin x}{x} \, dx = -\int_0^\infty e^{-\alpha x} \sin x \, dx
$$

3. Evaluate $\int_0^\infty e^{-\alpha x} \sin x \, dx$ using standard Laplace transform / double IBP:

$$
\int_0^\infty e^{-\alpha x} \sin x \, dx = \frac{1}{\alpha^2 + 1} \implies I'(\alpha) = -\frac{1}{\alpha^2 + 1}
$$

4. Integrate $I'(\alpha)$ with respect to $\alpha$:

$$
I(\alpha) = -\arctan(\alpha) + C
$$

5. Find $C$ by considering $\lim_{\alpha \to \infty} I(\alpha) = 0$:

$$
\lim_{\alpha \to \infty} (-\arctan(\alpha) + C) = -\frac{\pi}{2} + C = 0 \implies C = \frac{\pi}{2}
$$

6. Evaluate at $\alpha = 0$:

$$
I(0) = -\arctan(0) + \frac{\pi}{2} = \frac{\pi}{2}
$$

This step silently uses $\lim_{\alpha \to 0^+} I(\alpha) = I(0)$, i.e. continuity of $I$ at $\alpha = 0$, even though $I(0)$ is only conditionally convergent. This is justified by Dirichlet's test: on $[0,\infty)$, $\int_0^X \sin x\,dx$ is bounded uniformly in $X$ and $e^{-\alpha x}/x$ decreases monotonically to $0$ as $x \to \infty$ for every $\alpha \ge 0$, so the convergence of $\int_0^\infty e^{-\alpha x}\frac{\sin x}{x}\,dx$ is uniform in $\alpha$ on $[0, \infty)$ (Abel's uniform convergence theorem); a uniform limit of continuous functions of $\alpha$ is continuous, which licenses passing $\alpha \to 0^+$ inside the formula for $I(\alpha)$.

$$
\boxed{\frac{\pi}{2}}
$$

**Key takeaway.**
Exponential damping parameters $e^{-\alpha x}$ regularize non-absolutely convergent integrals, rendering them solvable via parametric differentiation.

In [38]:
print("Problem L3.7 - the Dirichlet integral")
for alpha in (1.0, 0.3, 0.05):
    val, _ = integrate.quad(lambda z, a=alpha: np.exp(-a * z) * np.sin(z) / z,
                            0.0, np.inf, limit=800)
    print(f"  I(alpha = {alpha}) versus pi/2 - arctan(alpha):")
    close(val, np.pi / 2 - np.arctan(alpha), tol=1e-6)
partial = integrate.quad(lambda z: np.sin(z) / z, 0.0, np.pi)[0]
partial += sum(integrate.quad(lambda z: np.sin(z) / z, k * np.pi, (k + 1) * np.pi)[0]
               for k in range(1, 400))
print("  truncated int_0^{400 pi} sin(x)/x dx:")
close(partial, np.pi / 2, tol=5e-3)

Problem L3.7 - the Dirichlet integral
  I(alpha = 1.0) versus pi/2 - arctan(alpha):
  computed 0.785398163397   boxed 0.785398163397   |diff| = 4.18e-13   OK
  I(alpha = 0.3) versus pi/2 - arctan(alpha):
  computed 1.279339532273   boxed 1.279339532317   |diff| = 4.37e-11   OK
  I(alpha = 0.05) versus pi/2 - arctan(alpha):
  computed 1.520837931355   boxed 1.520837931073   |diff| = 2.83e-10   OK
  truncated int_0^{400 pi} sin(x)/x dx:
  computed 1.570000553087   boxed 1.570796326795   |diff| = 7.96e-04   OK


True

### Problem L3.8 — A Floor-Function Integral as a Geometric Series
**Source Attribution:** Numerical Analysis / Series-Integral Techniques.

**Statement.**

Evaluate:

$$
I = \int_0^\infty \lfloor x \rfloor e^{-x} \, dx
$$

**Intuition.**
The floor function $\lfloor x \rfloor$ is piecewise constant on each integer subinterval $[n, n+1)$ where $\lfloor x \rfloor = n$. The integral splits into an infinite sum of definite integrals over unit intervals.

**Solution.**
Express $I$ as an infinite summation of sub-integrals:

$$
I = \sum_{n=0}^\infty \int_n^{n+1} \lfloor x \rfloor e^{-x} \, dx = \sum_{n=0}^\infty \int_n^{n+1} n e^{-x} \, dx
$$

Evaluate the inner integral for fixed $n$:

$$
\int_n^{n+1} n e^{-x} \, dx = n \Big[ -e^{-x} \Big]_n^{n+1} = n \left( e^{-n} - e^{-(n+1)} \right) = n e^{-n} \left( 1 - e^{-1} \right)
$$

Substitute back into the infinite sum:

$$
I = (1 - e^{-1}) \sum_{n=0}^\infty n e^{-n} = (1 - e^{-1}) \sum_{n=1}^\infty n (e^{-1})^n
$$

Use the arithmetico-geometric series formula $\sum_{n=1}^\infty n r^n = \frac{r}{(1-r)^2}$ for $r = e^{-1} \lt 1$:

$$
\sum_{n=1}^\infty n e^{-n} = \frac{e^{-1}}{(1 - e^{-1})^2}
$$

Multiply by $(1 - e^{-1})$:

$$
I = (1 - e^{-1}) \cdot \frac{e^{-1}}{(1 - e^{-1})^2} = \frac{e^{-1}}{1 - e^{-1}} = \frac{1}{e - 1}
$$

$$
\boxed{\frac{1}{e - 1}}
$$

**Key takeaway.**
Piecewise step functions convert continuous domain integrals into discrete geometric infinite series.

In [39]:
print("Problem L3.8 - int_0^inf floor(x) e^{-x} dx")
partial = sum(integrate.quad(lambda z, n=n: n * np.exp(-z), n, n + 1)[0]
              for n in range(0, 200))
close(partial, 1 / (np.e - 1))

Problem L3.8 - int_0^inf floor(x) e^{-x} dx
  computed 0.581976706869   boxed 0.581976706869   |diff| = 0.00e+00   OK


True

### Problem L3.9 — Bonnet's Second Mean Value Theorem
**Source Attribution:** Kaczor & Nowak, *Problems in Mathematical Analysis II*, Real Analysis.

**Statement.**

State and prove Bonnet's Form of the Second Mean Value Theorem for Integrals: If $f$ is integrable on $[a, b]$ and $g$ is monotonic non-increasing with $g(x) \ge 0$ on $[a, b]$, then there exists $\xi \in [a, b]$ such that $\int_a^b f(x) g(x) \, dx = g(a) \int_a^\xi f(x) \, dx$.

**Intuition.**
When $g(x)$ is positive and decreasing, it acts as a fading weighting function. Abel's summation lemma extended to continuous limits traps the weighted integral within scaled bounds of the accumulation function $F(x) = \int_a^x f(t) dt$.

**Solution.**
We prove the theorem under the simplifying (and commonly stated) extra hypothesis $g \in C^1[a,b]$. (The fully general statement for merely monotone $g$, which may have jump discontinuities, is not reachable by approximating $g$ with $C^1$ functions — a monotone function with jumps cannot be uniformly approximated by continuous ones — and instead requires a direct discrete Abel-summation argument on Riemann sums, passed to the limit; we do not reproduce that version here.)
1. Assume $g \in C^1[a, b]$ with $g'(x) \le 0$.
2. Define the accumulation function $F(x) = \int_a^x f(t) \, dt$, so $F(a) = 0$ and $F'(x) = f(x)$.
3. Apply integration by parts to $\int_a^b f(x) g(x) \, dx = \int_a^b F'(x) g(x) \, dx$:

$$
\int_a^b f(x) g(x) \, dx = \Big[ F(x) g(x) \Big]_a^b - \int_a^b F(x) g'(x) \, dx = F(b) g(b) - \int_a^b F(x) g'(x) \, dx
$$

4. Since $g'(x) \le 0$, $-g'(x) \ge 0$. By the First Mean Value Theorem for Integrals applied to continuous $F(x)$ with non-negative density $-g'(x)$:

$$
-\int_a^b F(x) g'(x) \, dx = F(\xi) \int_a^b (-g'(x)) \, dx = F(\xi) (g(a) - g(b))
$$

for some $\xi \in [a, b]$.
5. Substitute this back into the IBP identity:

$$
\begin{aligned}
\int_a^b f(x) g(x) \, dx &= F(b) g(b) + F(\xi) g(a) - F(\xi) g(b) \\
&= g(a) F(\xi) + g(b) (F(b) - F(\xi))
\end{aligned}
$$

6. If $g(b) = 0$, this simplifies immediately to $g(a) F(\xi) = g(a) \int_a^\xi f(x) dx$. Otherwise $g(a) \gt 0$ (since $g \ge 0$ and $g$ non-increasing with $g(b) \neq 0$), and we rewrite the right-hand side of step 5 by factoring out $g(a)$:

$$
g(a) F(\xi) + g(b)(F(b) - F(\xi)) = g(a)\left[ F(\xi) + \frac{g(b)}{g(a)}\big(F(b) - F(\xi)\big) \right]
$$

Since $0 \le g(b) \le g(a)$, the ratio $\lambda := g(b)/g(a)$ lies in $[0,1]$, so the bracketed quantity $F(\xi) + \lambda(F(b) - F(\xi))$ is a convex combination of $F(\xi)$ and $F(b)$ and therefore lies between them. As $F$ is continuous on $[a,b]$ (it is an integral of $f$), the Intermediate Value Theorem produces $\xi^\ast$ between $\xi$ and $b$ with $F(\xi^\ast)$ equal to this bracketed value. Hence:

$$
\int_a^b f(x) g(x) \, dx = g(a) \int_a^{\xi^\ast} f(x) \, dx
$$

$$
\boxed{\text{Proved: } \exists \xi \in [a, b] \text{ s.t. } \int_a^b f(x) g(x) dx = g(a) \int_a^\xi f(x) dx}
$$

**Key takeaway.**
The Second Mean Value Theorem decouples bounded monotonic weight functions from continuous integrand accumulation functions.

In [40]:
print("Problem L3.9 - Bonnet's theorem, exhibiting the point xi")
f_b = lambda z: np.cos(3 * z)
g_b = lambda z: np.exp(-z)                      # non-negative and decreasing
a_, b_ = 0.0, 2.0
lhs, _ = integrate.quad(lambda z: f_b(z) * g_b(z), a_, b_)
F_b = lambda xi: np.sin(3 * xi) / 3             # int_0^xi cos(3t) dt
target = lhs / g_b(a_)
grid = np.linspace(a_, b_, 200_001)
xi = float(grid[np.argmin(np.abs(F_b(grid) - target))])
print(f"  lhs = int f g = {lhs:.10f},  g(a) = {g_b(a_):.4f},  xi = {xi:.6f} in [{a_}, {b_}]")
close(g_b(a_) * F_b(xi), lhs, tol=1e-5)

Problem L3.9 - Bonnet's theorem, exhibiting the point xi


  lhs = int f g = 0.0756610756,  g(a) = 1.0000,  xi = 0.970870 in [0.0, 2.0]
  computed 0.075662283239   boxed 0.075661075553   |diff| = 1.21e-06   OK


True

### Problem L3.10 — A Putnam Logarithmic Integral by Reflection
**Source Attribution:** Putnam 2005 A5.

**Statement.**

Evaluate the definite integral:

$$
I = \int_0^1 \frac{\ln(1 + x)}{1 + x^2} \, dx
$$

**Intuition.**
Substitute $x = \tan \theta$ to convert the rational trigonometric denominator $1 + x^2$ into $1 + \tan^2 \theta = \sec^2 \theta$, which cancels $dx = \sec^2 \theta \, d\theta$.

**Solution.**
Let $x = \tan \theta \implies dx = \sec^2 \theta \, d\theta$.  
Limits: when $x = 0$, $\theta = 0$; when $x = 1$, $\theta = \frac{\pi}{4}$.

$$
I = \int_0^{\frac{\pi}{4}} \frac{\ln(1 + \tan \theta)}{1 + \tan^2 \theta} (\sec^2 \theta \, d\theta) = \int_0^{\frac{\pi}{4}} \ln(1 + \tan \theta) \, d\theta
$$

Apply King's Property: substitute $\theta = \frac{\pi}{4} - u \implies d\theta = -du$:

$$
I = \int_0^{\frac{\pi}{4}} \ln\left( 1 + \tan\left( \frac{\pi}{4} - u \right) \right) \, du
$$

Recall the angle difference formula $\tan(\frac{\pi}{4} - u) = \frac{1 - \tan u}{1 + \tan u}$:

$$
1 + \tan\left(\frac{\pi}{4} - u\right) = 1 + \frac{1 - \tan u}{1 + \tan u} = \frac{1 + \tan u + 1 - \tan u}{1 + \tan u} = \frac{2}{1 + \tan u}
$$

Substitute this simplified argument into the logarithm:

$$
I = \int_0^{\frac{\pi}{4}} \ln\left( \frac{2}{1 + \tan u} \right) \, du = \int_0^{\frac{\pi}{4}} \left( \ln 2 - \ln(1 + \tan u) \right) \, du
$$

Split the integrals:

$$
I = \int_0^{\frac{\pi}{4}} \ln 2 \, du - \int_0^{\frac{\pi}{4}} \ln(1 + \tan u) \, du = \ln 2 \left( \frac{\pi}{4} - 0 \right) - I
$$

Add $I$ to both sides:

$$
2I = \frac{\pi}{4} \ln 2 \implies I = \frac{\pi}{8} \ln 2
$$

$$
\boxed{\frac{\pi}{8} \ln 2}
$$

**Key takeaway.**
Trigonometric substitutions combined with angle addition identities convert non-linear logarithmic rational integrals into constant-integrand reflections.

In [41]:
print("Problem L3.10 - int_0^1 ln(1+x)/(1+x^2) dx")
val, _ = integrate.quad(lambda z: np.log(1 + z) / (1 + z**2), 0.0, 1.0)
close(val, np.pi * np.log(2) / 8)

Problem L3.10 - int_0^1 ln(1+x)/(1+x^2) dx
  computed 0.272198261288   boxed 0.272198261288   |diff| = 5.55e-17   OK


True